# Supervised Classical ML — Masterclass

The first notebook in the **classical ML algorithms series**. Goal: when an interviewer asks you to implement linear regression / a decision tree / gradient boosting from scratch, you've already done it. When they ask "when would you reach for SVM over logistic regression?", you have a calibrated answer. When they ask about hyperparameter tuning + cross-validation pitfalls, you can list the failure modes.

## What's covered (in execution order)
1. **Linear regression** — closed form + gradient descent + Ridge/Lasso/ElasticNet
2. **Logistic regression** — log-loss + GD from scratch + multi-class strategies
3. **k-Nearest Neighbors** — distance-weighted, KD-tree intuition
4. **Naive Bayes** — Gaussian, Multinomial, Bernoulli — when to use which
5. **SVM** — linear from-scratch (hinge loss), kernel trick conceptually
6. **Decision Tree** — CART from scratch (the canonical interview implementation)
7. **Random Forest** — bagging on the from-scratch tree
8. **Gradient Boosting** — residual fitting from scratch (the canonical second interview implementation)
9. **XGBoost / LightGBM / CatBoost** — usage + each's distinguishing innovations
10. **Pipeline + Cross-validation** — encoding decisions, scaler decisions, k-fold variants, leakage in CV, nested CV

## Per-algorithm template
Each section follows the same 7-element structure:
1. **One-paragraph story** — mental model, when to reach for it
2. **Core math** — the loss / split criterion / decision rule
3. **From-scratch NumPy implementation** (where reasonable)
4. **sklearn implementation** with hyperparameter walk-through
5. **Visualization** — decision boundary, feature importance, etc.
6. **Cheat table** — time complexity, assumptions, drawbacks, needs scaling, handles missing, interpretability
7. **Interview Q&A** with at least one trap question

## How to use this notebook
- Read top to bottom once.
- For interview prep, the **from-scratch implementations + cheat tables + interview Q&A** are the highest-leverage cells.
- For a real project, sections 1-9 are the algorithms, section 10 is the workflow that wraps them.

## Stack (May 2026)
- Python 3.11+
- pandas 3.0.x, numpy 2.4.x
- scikit-learn 1.8.x
- xgboost 3.x, lightgbm 4.x, catboost 1.x
- matplotlib + seaborn


## Setup

In [3]:
# Standard imports used throughout the notebook.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# Reproducibility: modern NumPy Generator API.
RNG = np.random.default_rng(seed=42)

# Display settings.
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
pd.set_option("display.precision", 4)

sns.set_theme(style="whitegrid", context="notebook")

# sklearn — confirm version.
import sklearn
print(f"numpy {np.__version__}  |  pandas {pd.__version__}  |  sklearn {sklearn.__version__}")

# Boosting libraries — confirm available.
import xgboost as xgb
import lightgbm as lgbm
import catboost as cb
print(f"xgboost {xgb.__version__}  |  lightgbm {lgbm.__version__}  |  catboost {cb.__version__}")


numpy 2.0.1  |  pandas 2.2.3  |  sklearn 1.6.1
xgboost 3.2.0  |  lightgbm 4.6.0  |  catboost 1.2.10


---
# 1. Linear regression

The model that everything else is "linear regression but with a twist." Logistic regression = linear regression with a sigmoid + log-loss. Neural networks = stacked linear regressions with non-linearities. Even a single decision tree's leaf prediction is a piecewise-constant approximation that linear regression upgrades to a piecewise-linear one.

If you can't write linear regression from memory in 3 minutes, **stop interview prep and do that first**.

## 1.1 Story

Predict a continuous target as a linear combination of features:
$$\hat{y} = w_0 + w_1 x_1 + w_2 x_2 + \dots + w_d x_d = \mathbf{w}^\top \mathbf{x}$$

Find $\mathbf{w}$ that minimizes **mean squared error**:
$$\mathcal{L}(\mathbf{w}) = \frac{1}{n} \sum_{i=1}^n (y_i - \mathbf{w}^\top \mathbf{x}_i)^2$$

## 1.2 Core math

Two ways to solve it.

**Normal equation (closed form).** Take the gradient, set to zero:
$$\nabla_{\mathbf{w}} \mathcal{L} = -\frac{2}{n} X^\top (y - X\mathbf{w}) = 0 \implies \boxed{\mathbf{w} = (X^\top X)^{-1} X^\top y}$$

Works when $X^\top X$ is invertible. Time complexity: $O(d^3)$ for the inverse plus $O(nd^2)$ for the matrix product. Fine for small $d$ (up to ~1000s of features). Infeasible for high-dimensional problems.

**Gradient descent.** Iterate:
$$\mathbf{w} \leftarrow \mathbf{w} - \eta \cdot \frac{-2}{n} X^\top (y - X\mathbf{w})$$

Time complexity per epoch: $O(nd)$. Scales to large $n$ and large $d$. Need to tune learning rate.

## 1.3 Assumptions (and what breaks when they fail)
1. **Linearity** — relationship between $X$ and $y$ is linear. Violation: model misspecified; residuals show curvature.
2. **Independence of errors** — Violation: time series or grouped data; standard errors wrong.
3. **Homoscedasticity** — residual variance constant across $\hat{y}$. Violation: heteroscedasticity; predictions still unbiased but inefficient and CIs wrong.
4. **Normality of errors** — Violation: matters for confidence intervals but not for point predictions.
5. **No perfect multicollinearity** — features not linearly dependent. Violation: $X^\top X$ singular, can't invert.

## 1.4 From-scratch implementation

In [4]:
# Generate a small synthetic regression problem to test against.
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression as SKLinReg

X_reg, y_reg = make_regression(
    n_samples=500, n_features=5, noise=10.0, random_state=42, coef=False
)
print(f"X shape: {X_reg.shape}, y shape: {y_reg.shape}")


X shape: (500, 5), y shape: (500,)


In [5]:
class LinearRegressionScratch:
    '''
    Linear regression solved via the normal equation (with pseudoinverse fallback).
    Equivalent to sklearn.linear_model.LinearRegression with fit_intercept=True.

    Time complexity:
      fit:     O(n*d^2 + d^3)
      predict: O(n*d)
    Memory: O(d^2) for X^T X
    '''
    def __init__(self, fit_intercept=True):
        self.fit_intercept = fit_intercept
        self.coef_ = None
        self.intercept_ = 0.0

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)

        # Augment X with a column of ones for the intercept.
        if self.fit_intercept:
            X_aug = np.hstack([np.ones((X.shape[0], 1)), X])
        else:
            X_aug = X

        # Use pseudoinverse — robust to singular X^T X (it handles collinearity gracefully).
        # np.linalg.pinv uses SVD under the hood, more numerically stable than np.linalg.inv.
        w = np.linalg.pinv(X_aug) @ y

        if self.fit_intercept:
            self.intercept_ = w[0]
            self.coef_ = w[1:]
        else:
            self.coef_ = w
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        return X @ self.coef_ + self.intercept_


# Train both, verify identical coefficients.
mine = LinearRegressionScratch().fit(X_reg, y_reg)
theirs = SKLinReg().fit(X_reg, y_reg)

print("My coefficients:    ", np.round(mine.coef_, 4))
print("sklearn coefficients:", np.round(theirs.coef_, 4))
print(f"Coefficients match? {np.allclose(mine.coef_, theirs.coef_)}")
print(f"Intercepts match?   {np.isclose(mine.intercept_, theirs.intercept_)}")


My coefficients:     [28.8045 81.5839 31.1    68.6941 10.6778]
sklearn coefficients: [28.8045 81.5839 31.1    68.6941 10.6778]
Coefficients match? True
Intercepts match?   True


## 1.5 Gradient descent variant (for when $d$ is huge)

In [ ]:
class LinearRegressionGD:
    '''
    Linear regression via mini-batch gradient descent.
    For when n*d is too large for the normal equation to be practical.

    Loss:  L = (1/n) * ||y - X w||^2
    Grad:  dL/dw = -(2/n) * X^T (y - X w)
    '''
    def __init__(self, lr=0.01, n_epochs=200, batch_size=32, fit_intercept=True):
        self.lr = lr
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        self.fit_intercept = fit_intercept
        self.history_ = []

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        n, d = X.shape

        if self.fit_intercept:
            X = np.hstack([np.ones((n, 1)), X])
            d += 1

        rng = np.random.default_rng(0)
        w = np.zeros(d)

        for epoch in range(self.n_epochs):
            idx = rng.permutation(n)
            for start in range(0, n, self.batch_size):
                batch = idx[start:start + self.batch_size]
                Xb, yb = X[batch], y[batch]
                grad = -(2.0 / len(yb)) * Xb.T @ (yb - Xb @ w)
                w -= self.lr * grad
            # Track loss per epoch.
            loss = np.mean((y - X @ w) ** 2)
            self.history_.append(loss)

        if self.fit_intercept:
            self.intercept_ = w[0]
            self.coef_ = w[1:]
        else:
            self.intercept_ = 0.0
            self.coef_ = w
        return self

    def predict(self, X):
        return np.asarray(X, dtype=float) @ self.coef_ + self.intercept_


# Note: standardize X for GD or learning rate has to be tiny.
from sklearn.preprocessing import StandardScaler
X_scaled = StandardScaler().fit_transform(X_reg)

gd = LinearRegressionGD(lr=0.05, n_epochs=100).fit(X_scaled, y_reg)
nm = LinearRegressionScratch().fit(X_scaled, y_reg)

print("GD final loss:    ", round(gd.history_[-1], 4))
print("Normal eq loss:   ", round(np.mean((y_reg - nm.predict(X_scaled))**2), 4))

# Loss should be decreasing.
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(gd.history_)
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE loss")
ax.set_title("Gradient descent convergence")
plt.show()


## 1.6 Regularized variants — Ridge, Lasso, ElasticNet

Plain OLS minimizes MSE. **Regularized** variants add a penalty on the weights:

| Variant | Penalty | Effect | Closed form? |
|---|---|---|---|
| **Ridge** | $\lambda \|\mathbf{w}\|_2^2$ | Shrinks all weights toward zero, never exactly zero | Yes: $\mathbf{w} = (X^\top X + \lambda I)^{-1} X^\top y$ |
| **Lasso** | $\lambda \|\mathbf{w}\|_1$ | Drives many weights to **exactly** zero — feature selection | No, needs coordinate descent |
| **ElasticNet** | $\lambda_1 \|\mathbf{w}\|_1 + \lambda_2 \|\mathbf{w}\|_2^2$ | Compromise; handles correlated features better than pure Lasso | No |

**When to use which:**
- Default to **Ridge** when you have many features and want stable coefficients.
- **Lasso** when you suspect most features are irrelevant and want automatic selection.
- **ElasticNet** when features are correlated (Lasso arbitrarily picks one of a correlated group; ElasticNet shares weight across them).

In [ ]:
# Build a dataset where some features are pure noise — see if Lasso zeros them out.
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet

X_noisy, y_noisy = make_regression(
    n_samples=500, n_features=20, n_informative=5,  # only 5 of 20 features matter
    noise=10.0, random_state=0,
)
X_noisy = StandardScaler().fit_transform(X_noisy)  # Lasso/Ridge need scaled features

models = {
    "OLS":        LinearRegression(),
    "Ridge α=1":  Ridge(alpha=1.0),
    "Lasso α=1":  Lasso(alpha=1.0),
    "ElasticNet": ElasticNet(alpha=1.0, l1_ratio=0.5),
}

coefs = {}
for name, m in models.items():
    m.fit(X_noisy, y_noisy)
    coefs[name] = m.coef_

coef_df = pd.DataFrame(coefs, index=[f"x{i}" for i in range(20)]).round(3)
print(coef_df)
print(f"\nLasso zero coefficients: {(coef_df['Lasso α=1'] == 0).sum()}/20")
print("Notice Lasso eliminates noise features automatically.")


In [ ]:
# Visualize coefficient shrinkage.
fig, ax = plt.subplots(figsize=(11, 4))
coef_df.plot(kind="bar", ax=ax)
ax.axhline(0, color="black", linewidth=0.5)
ax.set_title("Coefficient values: OLS vs Ridge vs Lasso vs ElasticNet")
ax.set_ylabel("Coefficient")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()


## 1.7 Feature scaling — when does linear regression need it?

| Variant | Needs scaling? | Why |
|---|---|---|
| OLS via normal equation | **No** | Solution is invariant to scaling (coefficients rescale, predictions unchanged) |
| OLS via gradient descent | **Yes** | GD takes uneven step sizes per feature; unscaled features → slow convergence |
| Ridge | **Yes** | The penalty $\|w\|^2$ is not scale-invariant — features with larger units get penalized more |
| Lasso | **Yes** | Same reason as Ridge |

**Rule of thumb:** always scale before regularized regression. Standardize (zero mean, unit variance) is the default.

## 1.8 Cheat table

In [6]:
linreg_cheat = pd.DataFrame([
    ["Linear regression (OLS)",   "O(nd² + d³)",     "O(d)",       "No (closed form)",  "No",                "High (coefficients = effect sizes)"],
    ["Ridge",                     "O(nd² + d³)",     "O(d)",       "Yes",               "No",                "High"],
    ["Lasso",                     "O(nd · iter)",    "O(d)",       "Yes",               "No",                "Very high (sparse coefs)"],
    ["ElasticNet",                "O(nd · iter)",    "O(d)",       "Yes",               "No",                "Very high"],
], columns=["Algorithm", "Train time", "Predict time", "Needs scaling?", "Handles missing?", "Interpretability"])
linreg_cheat


,Algorithm,Train time,Predict time,Needs scaling?,Handles missing?,Interpretability
0,Linear regression (OLS),O(nd² + d³),O(d),No (closed form),No,High (coefficients = effect sizes)
1,Ridge,O(nd² + d³),O(d),Yes,No,High
2,Lasso,O(nd · iter),O(d),Yes,No,Very high (sparse coefs)
3,ElasticNet,O(nd · iter),O(d),Yes,No,Very high


## 1.9 Interview Q&A

**Q: Why use the pseudoinverse instead of `np.linalg.inv` for the normal equation?**
> The pseudoinverse uses SVD under the hood, which gracefully handles singular or near-singular $X^\top X$ (collinear features). `inv` errors out or returns garbage. In production, always use `pinv` or `lstsq`.

**Q: What's the difference between Ridge and Lasso? When would you choose each?**
> Ridge ($L_2$ penalty) shrinks coefficients smoothly toward zero — good for **shrinkage** when many features matter slightly. Lasso ($L_1$ penalty) produces sparse solutions — coefficients are exactly zero — good for **feature selection** when you believe most features are irrelevant. Lasso fails when features are highly correlated (arbitrarily picks one of them); ElasticNet fixes this.

**Q (trap): If I add a perfectly correlated feature to a Ridge regression, what happens to the coefficients?**
> Ridge **splits the weight** between the two correlated features (each gets half). OLS would explode (singular matrix) — Ridge regularizes it. Lasso would arbitrarily pick **one** feature and zero out the other.

**Q: Why does gradient descent need feature scaling but the normal equation doesn't?**
> The MSE loss surface for unscaled features is a long, narrow ellipsoid. GD takes a step in the gradient direction, which oscillates across the narrow axis instead of progressing along the long axis. Scaling makes the ellipsoid spherical and GD converges fast. The normal equation directly jumps to the minimum — it doesn't care about the shape.

**Q (trap): Can OLS coefficients be negative even when all $y_i > 0$ and all $x_{ij} > 0$?**
> Yes. Coefficients are partial effects after controlling for other features. If two correlated features both increase $y$, one can get a negative coefficient because, conditional on the other, it adds nothing or has a residual negative association. This is "Simpson's paradox in regression."

**Q: When does R² equal exactly 1?**
> When the model fits the training data perfectly (zero residuals). On training data, R² = 1 means $X$ has at least as many columns as the rank of $y$ allows — usually overfitting. R² **can be negative** on held-out data when the model is worse than predicting the mean.


---
# 2. Logistic regression

Despite the name, it's a **classifier**. The default baseline for binary classification. Always run it first before reaching for anything fancier — if it gets 95% of the way there, the problem doesn't need a complex model.

## 2.1 Story

Predict the probability that $y = 1$ given $\mathbf{x}$:
$$P(y=1 \mid \mathbf{x}) = \sigma(\mathbf{w}^\top \mathbf{x}) = \frac{1}{1 + e^{-\mathbf{w}^\top \mathbf{x}}}$$

The sigmoid squashes the linear combination into $[0, 1]$. Decision rule: predict 1 if $\sigma(\mathbf{w}^\top \mathbf{x}) > 0.5$, else 0.

## 2.2 Core math

**Loss:** binary cross-entropy (a.k.a. log loss / negative log-likelihood):
$$\mathcal{L}(\mathbf{w}) = -\frac{1}{n} \sum_{i=1}^n \big[ y_i \log p_i + (1 - y_i) \log(1 - p_i) \big]$$
where $p_i = \sigma(\mathbf{w}^\top \mathbf{x}_i)$.

**Gradient:**
$$\nabla_{\mathbf{w}} \mathcal{L} = \frac{1}{n} X^\top (\mathbf{p} - \mathbf{y})$$

Beautifully simple — same structure as linear regression's gradient, but with $\mathbf{p}$ in place of $\hat{y}$. There's **no closed form** — solve via gradient descent or, in practice, **L-BFGS** (sklearn's default).

## 2.3 Why log loss, not squared error?

Squared error on classification probabilities (Brier score) is convex but has small gradients when predictions are far from labels — slow convergence. Log loss has **larger gradients** at extreme wrong predictions, pushing them back faster. Plus, log loss is the **maximum likelihood** loss under the Bernoulli model — there's a principled reason.

## 2.4 From-scratch implementation

In [ ]:
class LogisticRegressionScratch:
    '''
    Binary logistic regression via gradient descent.

    Loss:  L = -mean[y log p + (1-y) log(1-p)]  where p = sigmoid(X w)
    Grad:  dL/dw = (1/n) X^T (p - y)

    Time complexity:
      fit:     O(n * d * n_iter)
      predict: O(n * d)
    '''
    def __init__(self, lr=0.1, n_iter=1000, fit_intercept=True, l2=0.0):
        self.lr = lr
        self.n_iter = n_iter
        self.fit_intercept = fit_intercept
        self.l2 = l2
        self.history_ = []

    @staticmethod
    def _sigmoid(z):
        # Numerically stable: avoid overflow for large negative z by clipping.
        # For z >= 0: 1 / (1 + exp(-z));  for z < 0: exp(z) / (1 + exp(z))
        return np.where(z >= 0, 1.0 / (1.0 + np.exp(-z)), np.exp(z) / (1.0 + np.exp(z)))

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        n, d = X.shape
        if self.fit_intercept:
            X = np.hstack([np.ones((n, 1)), X])
            d += 1
        w = np.zeros(d)

        for _ in range(self.n_iter):
            z = X @ w
            p = self._sigmoid(z)
            grad = (X.T @ (p - y)) / n
            if self.l2 > 0:
                # L2 penalty: derivative is l2 * w. Don't regularize the intercept (w[0]).
                reg = self.l2 * w
                if self.fit_intercept:
                    reg[0] = 0
                grad += reg
            w -= self.lr * grad
            # Track log loss.
            eps = 1e-15
            loss = -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))
            self.history_.append(loss)

        if self.fit_intercept:
            self.intercept_ = w[0]
            self.coef_ = w[1:]
        else:
            self.intercept_ = 0.0
            self.coef_ = w
        return self

    def predict_proba(self, X):
        X = np.asarray(X, dtype=float)
        z = X @ self.coef_ + self.intercept_
        p = self._sigmoid(z)
        return np.column_stack([1 - p, p])

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X)[:, 1] >= threshold).astype(int)


# Validate against sklearn.
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression as SKLogReg

X_clf, y_clf = make_classification(
    n_samples=500, n_features=4, n_redundant=0, n_informative=4, random_state=42,
)
X_clf_s = StandardScaler().fit_transform(X_clf)

mine = LogisticRegressionScratch(lr=0.5, n_iter=2000).fit(X_clf_s, y_clf)
theirs = SKLogReg(C=1e10, max_iter=2000).fit(X_clf_s, y_clf)  # C=1e10 ~ no regularization

print("My coefficients:      ", np.round(mine.coef_, 3))
print("sklearn coefficients: ", np.round(theirs.coef_[0], 3))
print(f"Accuracy mine:    {(mine.predict(X_clf_s) == y_clf).mean():.3f}")
print(f"Accuracy sklearn: {(theirs.predict(X_clf_s) == y_clf).mean():.3f}")


## 2.5 Decision boundary visualization

In [ ]:
# Train logistic regression on 2D data so we can plot the decision boundary.
X_2d, y_2d = make_classification(
    n_samples=300, n_features=2, n_redundant=0, n_informative=2,
    n_clusters_per_class=1, random_state=1,
)

clf = SKLogReg().fit(X_2d, y_2d)

# Create mesh grid for plotting.
h = 0.02
x_min, x_max = X_2d[:, 0].min() - 0.5, X_2d[:, 0].max() + 0.5
y_min, y_max = X_2d[:, 1].min() - 0.5, X_2d[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
Z = clf.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)

fig, ax = plt.subplots(figsize=(8, 6))
contour = ax.contourf(xx, yy, Z, levels=20, cmap="RdBu_r", alpha=0.6)
ax.scatter(X_2d[:, 0], X_2d[:, 1], c=y_2d, edgecolors="black", cmap="RdBu_r", s=40)
plt.colorbar(contour, ax=ax, label="P(y=1)")
ax.set_title("Logistic regression decision boundary (probability landscape)")
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
plt.show()

# The decision boundary is the line where P(y=1) = 0.5, which is where w^T x = 0.
# Logistic regression always produces a LINEAR decision boundary in the input space.


**Critical interview point:** logistic regression's decision boundary is **linear** in the input features. It cannot learn an XOR pattern. If your data isn't linearly separable, either:
1. Add polynomial / interaction features (feature engineering), or
2. Reach for a model with non-linear decision boundaries (trees, SVM with RBF, neural net).

## 2.6 Multi-class strategies

In [ ]:
# For K > 2 classes, two approaches:
#
# 1. One-vs-Rest (OvR): train K binary classifiers, one per class.
#    Predict: max P(class k vs rest) over k.
#    Use OneVsRestClassifier wrapper in modern sklearn.
#
# 2. Multinomial (softmax): generalize sigmoid to softmax, train one model.
#    Predict: argmax over softmax outputs.
#    Default in sklearn 1.7+ — produces calibrated probabilities that sum to 1.

from sklearn.multiclass import OneVsRestClassifier

X_mc, y_mc = make_classification(
    n_samples=500, n_features=4, n_informative=4, n_redundant=0,
    n_classes=4, n_clusters_per_class=1, random_state=0,
)
X_mc_s = StandardScaler().fit_transform(X_mc)

# OvR: wrap a binary logistic regression in OneVsRestClassifier
ovr = OneVsRestClassifier(SKLogReg(max_iter=1000)).fit(X_mc_s, y_mc)
# Multinomial: this is the default for LogisticRegression in sklearn 1.7+
mnm = SKLogReg(max_iter=1000).fit(X_mc_s, y_mc)

print(f"OvR accuracy:           {ovr.score(X_mc_s, y_mc):.3f}")
print(f"Multinomial accuracy:   {mnm.score(X_mc_s, y_mc):.3f}")
print(f"\nOvR probabilities (first row, normalized to sum to 1): {ovr.predict_proba(X_mc_s[:1])[0].round(3)}")
print(f"Multinomial probs     (first row, sums to 1 exactly):    {mnm.predict_proba(X_mc_s[:1])[0].round(3)}")
print("\nNote: in sklearn 1.7+, the `multi_class` parameter was removed.")
print("Multinomial (softmax) is now the default; use OneVsRestClassifier for OvR.")


## 2.7 Cheat table

In [ ]:
logreg_cheat = pd.DataFrame([
    ["Logistic regression", "O(n · d · iter)", "O(n · d)", "Yes (for GD/regularized)", "No", "High (coefs = log-odds)"],
], columns=["Algorithm", "Train time", "Predict time", "Needs scaling?", "Handles missing?", "Interpretability"])

assumptions_table = pd.DataFrame([
    ["Linearity in log-odds", "log(p / (1-p)) is linear in features",
     "Add polynomial features or move to non-linear model"],
    ["Independence of observations", "Rows are independent samples",
     "For grouped/clustered data, use GEEs or mixed models"],
    ["No perfect multicollinearity", "Features not linearly dependent",
     "Drop one, or use L2 regularization (it tolerates collinearity)"],
    ["Large sample / well-balanced classes", "Especially needed for stable coefs",
     "For imbalanced data, use class_weight='balanced' or stratified sampling"],
], columns=["Assumption", "Statement", "Fix when violated"])

print("Cheat table:")
print(logreg_cheat)
print("\nAssumptions:")
print(assumptions_table.to_string(index=False))


## 2.8 Interview Q&A

**Q: Why is it called "logistic regression" if it's a classifier?**
> Historical naming. The model *is* a regression — it regresses the log-odds onto a linear function of features. Classification is what we *do* with the output, but the underlying machinery fits a regression. The name predates ML convention.

**Q: Can logistic regression learn XOR?**
> No. The decision boundary is linear in the input features. XOR requires a non-linear boundary. You can fake it by adding the interaction term $x_1 \cdot x_2$ — then logistic regression on $(x_1, x_2, x_1 x_2)$ *can* solve XOR. But "pure" logistic regression with raw features cannot.

**Q (trap): If two features are perfectly correlated, what happens to logistic regression coefficients?**
> Without regularization, the solver fails or returns unstable coefficients (the gradient is degenerate along the correlated direction). With L2 (Ridge-like), the weight is split between the two features. With L1 (Lasso-like), one of them gets zeroed out.

**Q: Why do we use log loss instead of accuracy as the training objective?**
> Accuracy is non-differentiable (it's a step function of the predictions). Log loss is differentiable and convex in $\mathbf{w}$, so we can use gradient methods. Also: log loss penalizes confident-wrong predictions much more harshly than uncertain-wrong, which is the right inductive bias.

**Q: What does the coefficient of a logistic regression feature mean?**
> $w_j$ is the **change in log-odds** of $y=1$ per unit increase in $x_j$, holding other features constant. Exponentiating gives the **odds ratio**: $e^{w_j}$ tells you how the odds multiply for a unit change. If $w_j = 0.7$, then $e^{0.7} \approx 2$ → a unit increase doubles the odds.

**Q (trap): I trained logistic regression on imbalanced data (99% class 0, 1% class 1) and got 99% accuracy. Is the model good?**
> Probably not. A model that predicts "0" always gets 99% accuracy on this data. Check **precision/recall/F1 for class 1** and **PR-AUC** — those are the diagnostics that matter for imbalanced classification. Also consider `class_weight='balanced'` to upweight the minority class during training.


---
# 3. k-Nearest Neighbors (kNN)

The simplest non-trivial algorithm. **No training, only inference.** For each test point, find the k closest training points by some distance metric, and predict the majority label (classification) or mean target (regression).

The model **is** the training data. That's both its strength (instantly adaptive to new data) and its curse (slow at inference, memory-heavy, bad with high dimensions).

## 3.1 Core math

Given test point $\mathbf{x}$ and training set $\{(\mathbf{x}_i, y_i)\}_{i=1}^n$:
1. Compute $d(\mathbf{x}, \mathbf{x}_i)$ for all $i$ (Euclidean is default; Manhattan, cosine, Minkowski also common)
2. Sort distances, pick the $k$ smallest indices $\mathcal{N}_k(\mathbf{x})$
3. **Classification:** $\hat{y} = \text{mode}(y_i : i \in \mathcal{N}_k(\mathbf{x}))$. Optional: weight votes by $1/d$.
4. **Regression:** $\hat{y} = \text{mean}(y_i : i \in \mathcal{N}_k(\mathbf{x}))$

## 3.2 The curse of dimensionality

As dimensionality grows, **all points become approximately equidistant** to each other. The ratio of farthest-neighbor distance to nearest-neighbor distance approaches 1. So "nearest" loses meaning. kNN's accuracy degrades sharply when $d > \sim 30$ unless you reduce dimensions first (PCA, embeddings).

## 3.3 From-scratch implementation

In [ ]:
class KNNScratch:
    '''
    Simple kNN classifier with Euclidean distance.

    Time complexity:
      fit:      O(1) — just stores the data
      predict:  O(n * d * m) for m test points  (brute force)
                With KD-tree: O(d * m * log(n))  (only helpful for low d)
                With Ball tree / HNSW: better for higher d
    Memory: O(n * d) for storage
    '''
    def __init__(self, k=5, weights="uniform"):
        self.k = k
        self.weights = weights   # "uniform" or "distance"

    def fit(self, X, y):
        self.X_train_ = np.asarray(X, dtype=float)
        self.y_train_ = np.asarray(y)
        return self

    def _pairwise_distances(self, X):
        # Vectorized squared Euclidean: ||a - b||^2 = ||a||^2 + ||b||^2 - 2 a·b
        a2 = (X ** 2).sum(axis=1, keepdims=True)            # (m, 1)
        b2 = (self.X_train_ ** 2).sum(axis=1, keepdims=True).T  # (1, n)
        ab = X @ self.X_train_.T                             # (m, n)
        dist2 = np.maximum(0, a2 + b2 - 2 * ab)              # clip floating-point noise
        return np.sqrt(dist2)

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        D = self._pairwise_distances(X)                      # (m, n)
        # argpartition is faster than full sort for finding top-k.
        idx = np.argpartition(D, self.k, axis=1)[:, :self.k]  # (m, k)

        preds = []
        for i, neigh in enumerate(idx):
            labels = self.y_train_[neigh]
            if self.weights == "distance":
                w = 1.0 / (D[i, neigh] + 1e-9)
                # Weighted vote per unique label.
                unique, _ = np.unique(labels, return_inverse=True)
                counts = np.array([w[labels == u].sum() for u in unique])
                preds.append(unique[counts.argmax()])
            else:
                # Plain majority vote.
                vals, cnts = np.unique(labels, return_counts=True)
                preds.append(vals[cnts.argmax()])
        return np.array(preds)


# Validate against sklearn.
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split

X_kn, y_kn = make_classification(n_samples=400, n_features=4, n_informative=4,
                                   n_redundant=0, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X_kn, y_kn, test_size=0.3, random_state=0)

mine = KNNScratch(k=5).fit(X_tr, y_tr)
theirs = KNeighborsClassifier(n_neighbors=5).fit(X_tr, y_tr)

print(f"My accuracy:      {(mine.predict(X_te) == y_te).mean():.3f}")
print(f"sklearn accuracy: {(theirs.predict(X_te) == y_te).mean():.3f}")
print(f"Predictions match exactly? {(mine.predict(X_te) == theirs.predict(X_te)).mean():.3f}")


## 3.4 Choosing k

- **Small k (e.g. k=1):** high variance — sensitive to noise, decision boundary is wiggly, prone to overfitting.
- **Large k:** high bias — smooths over local structure, decision boundary becomes too simple.
- **Rule of thumb:** start with $k = \sqrt{n}$, tune via cross-validation.
- **Choose k odd for binary classification** to avoid ties.

In [ ]:
# Plot the bias-variance tradeoff: accuracy vs k.
from sklearn.model_selection import cross_val_score

ks = [1, 3, 5, 7, 11, 15, 25, 51, 101]
scores = []
for k in ks:
    s = cross_val_score(KNeighborsClassifier(n_neighbors=k), X_kn, y_kn, cv=5).mean()
    scores.append(s)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ks, scores, marker="o")
ax.set_xscale("log")
ax.set_xlabel("k (log scale)")
ax.set_ylabel("5-fold CV accuracy")
ax.set_title("kNN — choosing k via cross-validation")
ax.grid(True, alpha=0.3)
plt.show()


## 3.5 Cheat table

In [ ]:
knn_cheat = pd.DataFrame([
    ["kNN (brute force)", "O(1)", "O(n · d · m)", "Yes (distance metric is scale-sensitive)",
     "No", "Low (no learned weights)"],
    ["kNN (KD-tree)",     "O(n · d · log n)", "O(d · m · log n) for low d", "Yes",
     "No", "Low"],
], columns=["Algorithm", "Train time", "Predict time", "Needs scaling?", "Handles missing?", "Interpretability"])
knn_cheat


## 3.6 Interview Q&A

**Q: Does kNN train?**
> Effectively no. The "training" step is just storing the data. All the work happens at prediction time. This is called a **lazy learner** (vs **eager learners** like logistic regression that do heavy compute upfront).

**Q: Why does kNN need feature scaling?**
> Euclidean distance is dominated by features with large numerical ranges. If salary is in lakhs (10⁵) and age is in years (10¹), distance is essentially determined by salary alone. Always standardize or normalize before kNN. This is the #1 kNN gotcha in interviews.

**Q (trap): Is kNN affected by the curse of dimensionality more than other models?**
> Yes, severely. As $d$ grows, distance concentration means the ratio (max distance / min distance) → 1, so "nearest" loses semantic meaning. kNN works well up to ~30 dimensions, breaks down past ~50. Models like trees and linear models with regularization tolerate high dimensions much better.

**Q: Why might you use `k=sqrt(n)` as a rule of thumb?**
> It balances the bias-variance tradeoff: as $n$ grows, $k$ grows but more slowly. Empirically works well; always tune with CV anyway.

**Q (trap): For a regression problem, what's the difference between kNN with uniform weights and distance weights?**
> Uniform: $\hat{y}$ = mean of the k neighbors' targets — every neighbor contributes equally. Distance-weighted: $\hat{y}$ = weighted mean with $w_i = 1/d_i$. Distance weighting reduces the influence of distant neighbors, smoother predictions near data, but **more sensitive to neighbors at exactly distance 0** (which can happen with duplicate rows — add a small epsilon).


---
# 4. Naive Bayes

Apply Bayes' rule + assume features are **conditionally independent given the class**. The "naive" assumption is almost always false in real data, yet the model often works surprisingly well — especially for text. Famously the spam-filter algorithm of the 2000s.

## 4.1 Core math

By Bayes' rule:
$$P(y = c \mid \mathbf{x}) = \frac{P(\mathbf{x} \mid y = c) P(y = c)}{P(\mathbf{x})}$$

The **naive** assumption: features are conditionally independent given the class.
$$P(\mathbf{x} \mid y = c) = \prod_{j=1}^d P(x_j \mid y = c)$$

So the prediction rule becomes:
$$\hat{y} = \arg\max_c \; P(y = c) \prod_{j=1}^d P(x_j \mid y = c)$$

Take logs for numerical stability:
$$\hat{y} = \arg\max_c \; \log P(y = c) + \sum_{j=1}^d \log P(x_j \mid y = c)$$

## 4.2 The three variants

| Variant | Feature type | $P(x_j \mid y = c)$ model | Use case |
|---|---|---|---|
| **GaussianNB** | Continuous | Gaussian with class-wise mean & variance | Numerical features (when feature distributions are roughly bell-shaped) |
| **MultinomialNB** | Counts (non-negative integers) | Multinomial | Text classification with TF or count features |
| **BernoulliNB** | Binary | Bernoulli | Text with presence/absence features, binary features |

## 4.3 GaussianNB from scratch

In [ ]:
class GaussianNBScratch:
    '''
    Gaussian Naive Bayes from scratch.

    For each class c and each feature j, model:
      P(x_j | y=c) ~ Normal(mu_jc, sigma_jc^2)

    Prediction is the argmax over classes of:
      log P(y=c) + sum_j log N(x_j | mu_jc, sigma_jc^2)

    Time complexity:
      fit:     O(n · d · C)  where C = number of classes
      predict: O(m · d · C)
    '''
    def __init__(self, var_smoothing=1e-9):
        self.var_smoothing = var_smoothing  # avoid divide-by-zero on constant features

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        n, d = X.shape

        # Per-class mean and variance for each feature.
        self.priors_ = np.zeros(len(self.classes_))
        self.means_  = np.zeros((len(self.classes_), d))
        self.vars_   = np.zeros((len(self.classes_), d))

        for k, c in enumerate(self.classes_):
            Xc = X[y == c]
            self.priors_[k] = len(Xc) / n
            self.means_[k]  = Xc.mean(axis=0)
            self.vars_[k]   = Xc.var(axis=0) + self.var_smoothing
        return self

    def _log_likelihood(self, X):
        # log P(x | y=c) = sum_j [-0.5 log(2 pi sigma^2) - 0.5 (x - mu)^2 / sigma^2]
        # Returns shape (n_samples, n_classes).
        n_samples = X.shape[0]
        n_classes = len(self.classes_)
        ll = np.zeros((n_samples, n_classes))
        for k in range(n_classes):
            mu, var = self.means_[k], self.vars_[k]
            log_pdf = -0.5 * (np.log(2 * np.pi * var) + ((X - mu) ** 2) / var)
            ll[:, k] = log_pdf.sum(axis=1)
        return ll

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        log_prior = np.log(self.priors_)                  # (n_classes,)
        log_post  = self._log_likelihood(X) + log_prior   # (n_samples, n_classes)
        return self.classes_[log_post.argmax(axis=1)]


# Validate.
from sklearn.naive_bayes import GaussianNB

X_nb, y_nb = make_classification(n_samples=500, n_features=4, n_informative=4,
                                   n_redundant=0, random_state=11)

mine = GaussianNBScratch().fit(X_nb, y_nb)
theirs = GaussianNB().fit(X_nb, y_nb)

print(f"My accuracy:      {(mine.predict(X_nb) == y_nb).mean():.3f}")
print(f"sklearn accuracy: {(theirs.predict(X_nb) == y_nb).mean():.3f}")
print(f"\nMy class priors:      {mine.priors_.round(3)}")
print(f"sklearn class priors: {theirs.class_prior_.round(3)}")


## 4.4 MultinomialNB on text

Multinomial NB is the canonical text classifier. Each document is a count vector (or TF-IDF) and we model the per-class word frequencies.

In [ ]:
# Use the 20-newsgroups corpus as a small text classification example.
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

# Build a tiny synthetic corpus for speed.
docs = [
    "free money click here win prize",        # spam
    "claim your free gift cash prize",        # spam
    "win lottery cash money instantly",       # spam
    "meeting tomorrow at 10am",                # ham
    "lunch with team on friday",               # ham
    "project deadline next week",              # ham
    "remember to submit the report",           # ham
    "your prize is waiting click link",        # spam
] * 30
labels = ([1, 1, 1, 0, 0, 0, 0, 1]) * 30

vec = CountVectorizer()
X_text = vec.fit_transform(docs)
print(f"Vectorized text shape: {X_text.shape}")
print(f"Vocabulary sample: {vec.get_feature_names_out()[:8]}")

mnb = MultinomialNB(alpha=1.0).fit(X_text, labels)   # alpha=1 is Laplace smoothing
print(f"\nMultinomialNB accuracy: {mnb.score(X_text, labels):.3f}")

# The model "log probability ratio" per class is interpretable: top spam words.
log_prob = mnb.feature_log_prob_   # shape (n_classes, n_features)
spam_minus_ham = log_prob[1] - log_prob[0]
top_spam_idx = np.argsort(spam_minus_ham)[-5:][::-1]
print("\nTop-5 spam-indicating words:")
for i in top_spam_idx:
    print(f"  {vec.get_feature_names_out()[i]:<12s} (log-ratio = {spam_minus_ham[i]:+.2f})")


## 4.5 Laplace smoothing

If a word never appears in a class during training, $P(\text{word} \mid \text{class}) = 0$ → the whole posterior collapses to zero. **Laplace (add-$\alpha$) smoothing** fixes this:
$$P(x_j \mid y = c) = \frac{N_{jc} + \alpha}{N_c + \alpha \cdot |V|}$$
where $N_{jc}$ is the count of feature $j$ in class $c$ and $|V|$ is the vocabulary size. $\alpha = 1$ is "Laplace smoothing"; smaller $\alpha$ (e.g., 0.01) is "Lidstone smoothing."

**Without smoothing, MultinomialNB is brittle to unseen words in test data. Always set `alpha > 0`** (sklearn defaults to 1.0).

## 4.6 Cheat table

In [ ]:
nb_cheat = pd.DataFrame([
    ["GaussianNB",     "O(n · d · C)", "O(m · d · C)", "No (just affects implicit Gaussian fit)",
     "No",  "Medium"],
    ["MultinomialNB",  "O(n · d · C)", "O(m · d · C)", "No (counts have natural scale)",
     "No",  "High (per-feature log-probs interpretable)"],
    ["BernoulliNB",    "O(n · d · C)", "O(m · d · C)", "No",
     "No",  "High"],
], columns=["Algorithm", "Train time", "Predict time", "Needs scaling?", "Handles missing?", "Interpretability"])
nb_cheat


## 4.7 Interview Q&A

**Q: Why does naive Bayes work surprisingly well even though the independence assumption is almost always violated?**
> Two reasons: (1) **classification doesn't need calibrated probabilities** — it just needs the right class to score highest. The independence assumption can give wrong absolute probabilities but still correct **rankings**. (2) The model has very few parameters (per-class mean and variance per feature), so it's hard to overfit — beating it requires either more data or a much more flexible model.

**Q: When does naive Bayes fail badly?**
> When features are strongly correlated and the correct decision requires their interaction (XOR-like). Also when a feature has zero probability in a class (without smoothing). For text, NB beats logistic regression with very small training sets but loses as training size grows.

**Q (trap): Should I use GaussianNB or MultinomialNB for TF-IDF features?**
> MultinomialNB, despite TF-IDF being float, not count. Sklearn's MultinomialNB internally treats them as a non-negative weighted count distribution and it works well in practice. GaussianNB assumes features are bell-shaped, which TF-IDF (with mass near zero) is not.

**Q: Why take logs in the prediction rule?**
> Two reasons: (1) **numerical underflow** — multiplying many small probabilities together gives near-zero floats that lose precision. (2) Log turns the product into a sum, which is faster and more stable.

**Q (trap): What's the difference between MultinomialNB and BernoulliNB on text?**
> MultinomialNB uses **word counts** (how many times each word appears). BernoulliNB uses **word presence/absence** (binary 0/1 per word). BernoulliNB explicitly penalizes the **absence** of words that are common in a class — different inductive bias. For short documents (tweets), BernoulliNB sometimes wins.


---
# 5. Support Vector Machines (SVM)

The "find the maximum-margin separating hyperplane" classifier. Used to be the SOTA before deep learning ate everything. Still relevant when you have small, clean datasets with clear margins, or when interpretability of the decision boundary matters.

## 5.1 Core math

**Linear SVM (hard margin).** Find the hyperplane $\mathbf{w}^\top \mathbf{x} + b = 0$ that separates the two classes with **maximum margin** (largest distance to the nearest point of either class).

The margin is $2 / \|\mathbf{w}\|$, so maximizing the margin means **minimizing $\|\mathbf{w}\|^2$**.

The optimization problem:
$$\min_{\mathbf{w}, b} \; \frac{1}{2} \|\mathbf{w}\|^2 \quad \text{s.t.} \quad y_i (\mathbf{w}^\top \mathbf{x}_i + b) \geq 1 \; \forall i$$

**Soft margin** (real data is noisy): allow some violations with slack variables $\xi_i$:
$$\min_{\mathbf{w}, b, \xi} \; \frac{1}{2} \|\mathbf{w}\|^2 + C \sum_i \xi_i \quad \text{s.t.} \quad y_i (\mathbf{w}^\top \mathbf{x}_i + b) \geq 1 - \xi_i, \; \xi_i \geq 0$$

Equivalent unconstrained form using **hinge loss**:
$$\min_{\mathbf{w}, b} \; \frac{1}{2} \|\mathbf{w}\|^2 + C \sum_i \max(0, 1 - y_i (\mathbf{w}^\top \mathbf{x}_i + b))$$

The hinge loss is zero for correctly classified points outside the margin, linearly penalized inside.

## 5.2 The kernel trick

Real data isn't always linearly separable. The kernel trick: **implicitly map** $\mathbf{x}$ to a higher-dimensional space $\phi(\mathbf{x})$ via a kernel function $K(\mathbf{x}_i, \mathbf{x}_j) = \phi(\mathbf{x}_i)^\top \phi(\mathbf{x}_j)$.

The dual SVM optimization only uses $\mathbf{x}_i^\top \mathbf{x}_j$, so we can swap in any valid kernel without ever explicitly computing $\phi$.

| Kernel | $K(\mathbf{x}_i, \mathbf{x}_j)$ | Use case |
|---|---|---|
| Linear | $\mathbf{x}_i^\top \mathbf{x}_j$ | High-dimensional, sparse (text) |
| Polynomial | $(\gamma \mathbf{x}_i^\top \mathbf{x}_j + r)^d$ | Moderate non-linearity |
| **RBF (Gaussian)** | $\exp(-\gamma \|\mathbf{x}_i - \mathbf{x}_j\|^2)$ | Default for general non-linear problems |
| Sigmoid | $\tanh(\gamma \mathbf{x}_i^\top \mathbf{x}_j + r)$ | Rarely used |

## 5.3 From-scratch implementation (linear SVM via SGD on hinge loss)

In [ ]:
class LinearSVMScratch:
    '''
    Linear SVM trained via sub-gradient descent on the hinge loss.

    Loss:  L = (1/2) * ||w||^2 + C * sum_i max(0, 1 - y_i * (w^T x_i + b))
    Gradient is a subgradient (hinge loss isn't differentiable at the boundary).

    Time complexity:
      fit:     O(n · d · n_iter)
      predict: O(n · d)
    '''
    def __init__(self, C=1.0, n_iter=1000, lr=0.01):
        self.C = C
        self.n_iter = n_iter
        self.lr = lr
        self.history_ = []

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.where(y == 0, -1, 1).astype(float)   # convert 0/1 to -1/+1 for SVM
        n, d = X.shape
        w = np.zeros(d)
        b = 0.0

        for it in range(self.n_iter):
            # Compute margins for all points.
            margins = y * (X @ w + b)                # (n,)
            # Subgradient of hinge loss: -y x where margin < 1, else 0.
            violators = margins < 1
            grad_w = w - self.C * (y[violators][:, None] * X[violators]).sum(axis=0)
            grad_b = -self.C * y[violators].sum()
            w -= self.lr * grad_w / n
            b -= self.lr * grad_b / n
            # Track loss.
            loss = 0.5 * np.sum(w**2) + self.C * np.maximum(0, 1 - margins).sum()
            self.history_.append(loss)

        self.coef_ = w
        self.intercept_ = b
        return self

    def decision_function(self, X):
        return np.asarray(X, dtype=float) @ self.coef_ + self.intercept_

    def predict(self, X):
        return (self.decision_function(X) >= 0).astype(int)


# Validate.
from sklearn.svm import LinearSVC

X_sv, y_sv = make_classification(n_samples=400, n_features=4, n_informative=4,
                                   n_redundant=0, random_state=2)
X_sv = StandardScaler().fit_transform(X_sv)

mine = LinearSVMScratch(C=1.0, n_iter=2000, lr=0.01).fit(X_sv, y_sv)
theirs = LinearSVC(C=1.0, max_iter=5000).fit(X_sv, y_sv)

print(f"My accuracy:      {(mine.predict(X_sv) == y_sv).mean():.3f}")
print(f"sklearn accuracy: {(theirs.predict(X_sv) == y_sv).mean():.3f}")


## 5.4 Kernel SVM with sklearn

In [ ]:
# Demonstrate non-linear classification with RBF kernel SVM.
from sklearn.datasets import make_moons
from sklearn.svm import SVC

X_moon, y_moon = make_moons(n_samples=300, noise=0.2, random_state=0)

# Try three kernels.
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, svm) in zip(axes, [
    ("Linear",       SVC(kernel="linear", C=1.0)),
    ("Polynomial",   SVC(kernel="poly", degree=3, C=1.0, gamma="scale")),
    ("RBF",          SVC(kernel="rbf", C=1.0, gamma="scale")),
]):
    svm.fit(X_moon, y_moon)
    # Plot decision boundary.
    h = 0.02
    x_min, x_max = X_moon[:, 0].min() - 0.5, X_moon[:, 0].max() + 0.5
    y_min, y_max = X_moon[:, 1].min() - 0.5, X_moon[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = svm.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="RdBu_r")
    ax.scatter(X_moon[:, 0], X_moon[:, 1], c=y_moon, edgecolors="black", s=25, cmap="RdBu_r")
    ax.set_title(f"{name} SVM (acc={svm.score(X_moon, y_moon):.3f})")
plt.tight_layout()
plt.show()


**Read the figure:** linear SVM gets ~85% on this two-moons problem (the boundary literally can't curve). Polynomial and RBF both capture the curvature. RBF is the standard default; it has a single tunable parameter `gamma` controlling the locality of the kernel.

## 5.5 The C and gamma knobs

- **C (regularization)** — larger C → softer margin penalty → fits training data more aggressively → risk of overfitting. Smaller C → wider margin → simpler model. Default `C=1.0`.
- **gamma (RBF only)** — controls how far the influence of a single training point reaches. Large gamma → narrow Gaussian → each point only influences nearby region → wiggly decision boundary, overfitting. Small gamma → wide Gaussian → smoother boundary.

Both are tuned via cross-validation. Common grid: `C in [0.1, 1, 10, 100]`, `gamma in ['scale', 0.001, 0.01, 0.1, 1]`.

## 5.6 Support vectors

After training, only a subset of training points have non-zero $\alpha$ (the dual variable) — the **support vectors**. These are points on or inside the margin. The decision boundary is defined entirely by them; removing non-support-vector training points wouldn't change the model.

This is why SVMs scale poorly with $n$: the worst case for SMO (sklearn's algorithm) is $O(n^3)$. For $n > 100{,}000$, switch to **LinearSVC** (which uses a linear-time solver) or to a different model class entirely (linear or boosted trees).

## 5.7 Cheat table

In [ ]:
svm_cheat = pd.DataFrame([
    ["Linear SVM (LinearSVC)",  "O(n · d · iter)",      "O(d)",          "Yes",
     "No",  "High (interpretable linear weights)"],
    ["Kernel SVM (SVC, RBF)",   "O(n² · d) to O(n³ · d)", "O(n_SV · d)",  "Yes (RBF is distance-based)",
     "No",  "Low (decision is sum over support vectors)"],
], columns=["Algorithm", "Train time", "Predict time", "Needs scaling?", "Handles missing?", "Interpretability"])
svm_cheat


## 5.8 Interview Q&A

**Q: What does "maximum margin" mean and why do we want it?**
> The margin is the perpendicular distance from the decision hyperplane to the nearest training point. Maximizing it gives the boundary the most "room" — intuitively, it generalizes better because small perturbations of test points won't push them across. There's also a formal generalization bound (Vapnik-Chervonenkis) that says wider margins → tighter generalization error bounds.

**Q: What's the kernel trick, in plain English?**
> Implicitly map data to a higher (possibly infinite) dimensional space where it's linearly separable, without ever explicitly computing the mapping. The "trick" is that the SVM only uses dot products $\mathbf{x}_i^\top \mathbf{x}_j$ in its dual form, so we can replace them with any valid kernel $K(\mathbf{x}_i, \mathbf{x}_j)$.

**Q (trap): Does SVM need feature scaling?**
> Almost always yes. RBF kernel is distance-based, so unscaled features (where some have huge ranges) dominate distance. Linear SVM also benefits because the hinge loss penalty couples features — large-scale features force large gradient updates. Always standardize before SVM.

**Q: SVM doesn't output probabilities. How do you get probability estimates?**
> Use Platt scaling: fit a logistic regression on the SVM's decision function values vs true labels (with cross-validation). sklearn's `SVC(probability=True)` does this internally — but warning, it adds significant training time. For ranking purposes, the raw `decision_function` output is usually sufficient.

**Q (trap): I trained an SVM on a 10M-row dataset and it's taking 8 hours. What went wrong?**
> SVM training scales between $O(n^2)$ and $O(n^3)$ for kernel SVMs. 10M rows is too many. Switch to **LinearSVC** (linear-time solver, $O(n d)$), or to logistic regression with SGD (scales the same), or to a tree-based model. As a rule of thumb, kernel SVM is competitive up to ~100K samples; beyond that, use other methods.

**Q: What are support vectors?**
> Training points that lie on the margin boundary (or inside it, for soft-margin). They have non-zero coefficients in the dual solution. The decision boundary is defined entirely by them — discarding non-support-vector training points wouldn't change the trained model.


---
# 6. Decision Tree (CART)

The canonical interview implementation. If you can write a decision tree from scratch in 20 minutes, you can answer the hardest non-LeetCode interview question in ML.

A decision tree partitions the feature space into rectangular regions by **recursive binary splits**. At each node, find the (feature, threshold) split that maximally reduces impurity. Stop when a stopping criterion is met. Predict by traversing root → leaf and returning the leaf's majority class (classification) or mean target (regression).

## 6.1 Core math

**Impurity for classification:** at a node with class proportions $\mathbf{p} = (p_1, \dots, p_K)$:

- **Gini impurity:** $G(\mathbf{p}) = 1 - \sum_k p_k^2 = \sum_k p_k (1 - p_k)$. Range $[0, 1 - 1/K]$.
- **Entropy:** $H(\mathbf{p}) = -\sum_k p_k \log_2 p_k$. Range $[0, \log_2 K]$.

Both measure "how mixed" the class labels are. Pure node → 0. Maximally mixed → max value. They give similar splits in practice; Gini is faster (no log).

**Impurity for regression:** **MSE** at a node, with prediction = node mean.

**Information gain** (how much the split reduces impurity):
$$\text{IG} = I(\text{parent}) - \frac{n_L}{n} I(\text{left child}) - \frac{n_R}{n} I(\text{right child})$$

**Best split** at a node = (feature, threshold) that maximizes IG.

## 6.2 The CART algorithm

```
function build_tree(X, y, depth):
    if stop_criterion(X, y, depth):  return Leaf(majority_or_mean(y))
    best_feature, best_threshold = find_best_split(X, y)
    if best_feature is None:          return Leaf(...)   # no useful split
    left  = build_tree(X[mask], y[mask], depth+1)
    right = build_tree(X[~mask], y[~mask], depth+1)
    return Node(best_feature, best_threshold, left, right)
```

Stopping criteria (any of):
- `depth >= max_depth`
- `n_samples < min_samples_split`
- All samples in this node have the same label
- Best split has zero information gain

## 6.3 From-scratch implementation

In [ ]:
class DecisionTreeClassifierScratch:
    '''
    Classification tree built via CART with Gini impurity.

    Time complexity:
      fit:     O(n · d · log n)  on average  (each split candidate is O(n), there are
               O(d · n) candidates per node, tree has ~log n levels with linear total work)
      predict: O(log n) per sample on a balanced tree, O(n) worst case
    '''
    def __init__(self, max_depth=None, min_samples_split=2, min_samples_leaf=1):
        self.max_depth = max_depth if max_depth is not None else float("inf")
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf

    @staticmethod
    def _gini(y):
        '''Gini impurity. y is a 1-D array of class labels.'''
        if len(y) == 0:
            return 0.0
        _, counts = np.unique(y, return_counts=True)
        p = counts / counts.sum()
        return 1.0 - (p ** 2).sum()

    def _best_split(self, X, y):
        '''Find the (feature, threshold) split that maximally reduces Gini.'''
        n, d = X.shape
        best = {"gain": 0.0, "feature": None, "threshold": None}
        parent_gini = self._gini(y)

        for j in range(d):
            # Candidate thresholds: midpoints between consecutive sorted unique values.
            sorted_vals = np.sort(np.unique(X[:, j]))
            if len(sorted_vals) < 2:
                continue
            thresholds = (sorted_vals[:-1] + sorted_vals[1:]) / 2

            for t in thresholds:
                left_mask = X[:, j] <= t
                right_mask = ~left_mask
                n_l, n_r = left_mask.sum(), right_mask.sum()
                if n_l < self.min_samples_leaf or n_r < self.min_samples_leaf:
                    continue
                gain = parent_gini - (n_l / n) * self._gini(y[left_mask]) \
                                   - (n_r / n) * self._gini(y[right_mask])
                if gain > best["gain"]:
                    best = {"gain": gain, "feature": j, "threshold": t}
        return best

    def _build(self, X, y, depth):
        # Stopping criteria.
        n = len(y)
        if (depth >= self.max_depth or
            n < self.min_samples_split or
            len(np.unique(y)) == 1):
            # Leaf: return a dict with the majority class.
            vals, counts = np.unique(y, return_counts=True)
            return {"leaf": True, "prediction": vals[counts.argmax()]}

        split = self._best_split(X, y)
        if split["feature"] is None or split["gain"] == 0:
            vals, counts = np.unique(y, return_counts=True)
            return {"leaf": True, "prediction": vals[counts.argmax()]}

        # Recurse on left and right.
        mask = X[:, split["feature"]] <= split["threshold"]
        return {
            "leaf": False,
            "feature": split["feature"],
            "threshold": split["threshold"],
            "left":  self._build(X[mask], y[mask], depth + 1),
            "right": self._build(X[~mask], y[~mask], depth + 1),
        }

    def fit(self, X, y):
        self.tree_ = self._build(np.asarray(X, dtype=float), np.asarray(y), depth=0)
        return self

    def _predict_one(self, x, node):
        while not node["leaf"]:
            if x[node["feature"]] <= node["threshold"]:
                node = node["left"]
            else:
                node = node["right"]
        return node["prediction"]

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        return np.array([self._predict_one(x, self.tree_) for x in X])


# Validate against sklearn.
from sklearn.tree import DecisionTreeClassifier

X_dt, y_dt = make_classification(n_samples=400, n_features=6, n_informative=4,
                                   n_redundant=0, random_state=3)
X_tr, X_te, y_tr, y_te = train_test_split(X_dt, y_dt, test_size=0.3, random_state=0)

mine = DecisionTreeClassifierScratch(max_depth=5).fit(X_tr, y_tr)
theirs = DecisionTreeClassifier(max_depth=5, criterion="gini", random_state=0).fit(X_tr, y_tr)

print(f"My accuracy:      {(mine.predict(X_te) == y_te).mean():.3f}")
print(f"sklearn accuracy: {(theirs.predict(X_te) == y_te).mean():.3f}")


Note: predictions may not match exactly because:
- Sklearn uses an optimized C implementation with different tie-breaking.
- We try thresholds at midpoints between consecutive unique values; sklearn does the same but in a different order.
- Accuracies should be very close (within ~1%) on most datasets.

## 6.4 Visualize the decision boundary

In [ ]:
# Compare decision boundary for varying tree depth.
from sklearn.tree import DecisionTreeClassifier

X_2d, y_2d = make_moons(n_samples=300, noise=0.25, random_state=0)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, depth in zip(axes, [1, 3, 7, 15]):
    clf = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X_2d, y_2d)
    h = 0.02
    x_min, x_max = X_2d[:, 0].min() - 0.5, X_2d[:, 0].max() + 0.5
    y_min, y_max = X_2d[:, 1].min() - 0.5, X_2d[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="RdBu_r")
    ax.scatter(X_2d[:, 0], X_2d[:, 1], c=y_2d, edgecolors="black", s=25, cmap="RdBu_r")
    ax.set_title(f"max_depth = {depth}\ntrain_acc = {clf.score(X_2d, y_2d):.3f}")
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()


**Read it:** depth=1 is a single split, brutal underfit. Depth=3 is reasonable. Depth=15 has carved out individual training points → overfitting. Notice the **axis-aligned, rectangular** boundary — that's the unavoidable signature of trees. Diagonal patterns require lots of small splits to approximate.

## 6.5 Feature importance and tree depth

For a single tree, **feature importance** = total impurity reduction summed across all splits that use that feature, weighted by the number of samples reaching each split. sklearn exposes this as `tree.feature_importances_`.

This metric is biased toward high-cardinality features (they have more candidate split points). For more robust importance, use **permutation importance** or **SHAP values**.

In [ ]:
# Demonstrate feature importance.
from sklearn.tree import DecisionTreeClassifier

clf = DecisionTreeClassifier(max_depth=5, random_state=0).fit(X_dt, y_dt)
imp = pd.Series(clf.feature_importances_, index=[f"x{i}" for i in range(X_dt.shape[1])]).sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
imp.plot.barh(ax=ax, color="steelblue")
ax.set_xlabel("Feature importance (impurity reduction)")
ax.set_title("Decision Tree feature importance")
plt.tight_layout()
plt.show()


## 6.6 Hyperparameters that control overfitting

| Hyperparameter | What it does | Increase → | Decrease → |
|---|---|---|---|
| `max_depth` | Maximum tree depth | More flexibility, risk overfit | Force simpler tree |
| `min_samples_split` | Min samples to attempt a split | Simpler tree | More flexibility |
| `min_samples_leaf` | Min samples per leaf | Smoother boundaries | Sharper boundaries |
| `max_features` | Features considered per split | Less randomness | More randomness (better for forests) |
| `min_impurity_decrease` | Min gain required for a split | Simpler tree | More splits |
| `ccp_alpha` | Cost-complexity pruning param | Aggressive pruning | Less pruning |

**Practical default:** `max_depth=5-8` for a single tree, then tune via CV.

## 6.7 Cheat table

In [ ]:
dt_cheat = pd.DataFrame([
    ["Decision Tree (CART)", "O(n · d · log n)", "O(log n) avg, O(n) worst",
     "No", "Yes (built-in surrogate splits)", "Very high (visualizable)"],
], columns=["Algorithm", "Train time", "Predict time", "Needs scaling?", "Handles missing?", "Interpretability"])
dt_cheat


## 6.8 Interview Q&A

**Q: Why don't decision trees need feature scaling?**
> Trees split on **thresholds within a single feature** — `x_j <= t`. Whether `x_j` is in millions or thousandths is irrelevant; the threshold scales with it. This is unique to trees and is one reason they're so popular in practice.

**Q: Why is a single decision tree usually a bad final model?**
> High variance. A small change in training data can produce a completely different tree (especially the top splits — once those change, everything downstream changes). Single trees overfit easily. Random forests and gradient boosting solve this by averaging or sequentially correcting many trees.

**Q (trap): Gini vs Entropy — which is better?**
> They give very similar splits in practice (the trees are usually nearly identical). Gini is faster because it skips the log. Entropy can be slightly more sensitive to minority classes. Don't tune this — pick Gini, move on. Interviewers who push on this are testing whether you know it doesn't matter.

**Q: How do trees handle missing values?**
> Two main strategies: (1) **surrogate splits** — at each split, store a backup feature that correlates with the primary one; if the primary is missing, use the surrogate (CART's original approach, sklearn doesn't do this). (2) **Sentinel value** — replace NaN with a number outside the feature's range, treat as its own branch (LightGBM, XGBoost). (3) **Default direction** — the tree learns which side missing values should go to (XGBoost's default).

**Q: What's the depth complexity of training a tree?**
> $O(n \cdot d \cdot \log n)$ for balanced trees in the average case. Each level of the tree processes $n$ total samples; at each split, you try $O(n)$ thresholds across $d$ features; there are $O(\log n)$ levels. Worst case (very skewed splits) is $O(n^2 \cdot d)$.

**Q (trap): I trained a tree with `max_depth=None` and it got 100% training accuracy. Why is this bad?**
> Memorization. Without depth limit, the tree grows until every leaf has one sample — it perfectly fits training data but generalizes terribly. Test accuracy will be much lower. Always set `max_depth` (or `min_samples_leaf`, `ccp_alpha`) and tune via CV.

**Q: Why are decision trees not great for predicting continuous targets with smooth relationships?**
> Trees produce **piecewise-constant** predictions — every leaf outputs a single value. For a smooth target like `y = sin(x)`, the tree's predictions are a staircase approximation. To get a smooth approximation you need many leaves (overfitting risk) or move to linear/spline/GAM models.


---
# 7. Random Forest

**Bagging on trees + random feature subsampling.** Reduces the high variance of a single tree by averaging many trees, each grown on a bootstrap sample of the data and considering only a random subset of features at each split.

The default "I don't know what model to try yet" baseline. Robust, doesn't need scaling, handles mixed feature types, gives feature importance, hard to break. Often within 1-3% of the optimal model with almost no tuning.

## 7.1 The recipe

```
for t = 1 to T:
    Sample n rows from training data WITH REPLACEMENT  (bootstrap)
    Build a tree on this sample, but at each split consider only a random
        subset of m features (typically m = sqrt(d) for classification, d/3 for regression)
    Don't prune.

Prediction: majority vote (classification) or mean (regression) over all T trees.
```

Two sources of randomness:
1. **Bootstrap sampling of rows** — each tree sees a different ~63% of the data (the rest are "out-of-bag" samples).
2. **Random feature subsampling per split** — decorrelates trees, prevents them all from picking the same dominant feature.

The second is what makes Random Forest more than just "many trees." Bagging alone (no feature subsampling) wouldn't decorrelate trees enough to reduce variance much.

## 7.2 Why does it work? — the variance reduction math

If we have $T$ trees each with variance $\sigma^2$ and average correlation $\rho$ between any two trees' predictions, the variance of the averaged prediction is:
$$\text{Var}(\bar{f}) = \rho \sigma^2 + \frac{1 - \rho}{T} \sigma^2$$

As $T \to \infty$, $\text{Var}(\bar{f}) \to \rho \sigma^2$. So we want **$\rho$ small** — decorrelated trees. Random feature subsampling reduces $\rho$. Bagging alone reduces $\rho$ only slightly because all trees still see most of the data and pick similar top splits.

## 7.3 From-scratch implementation

In [ ]:
class RandomForestClassifierScratch:
    '''
    Random Forest classifier built on top of our DecisionTreeClassifierScratch.

    For each of n_trees trees:
      1. Bootstrap sample (n samples WITH replacement from training data)
      2. Build a tree, randomly subsetting features at each split
         (we approximate by globally subsetting features per tree — close to true RF
          since each tree still sees a different subset)

    Time complexity:
      fit:     O(T · n · sqrt(d) · log n)
      predict: O(T · log n)
    Memory: O(T · tree_size)
    '''
    def __init__(self, n_trees=20, max_depth=None, max_features="sqrt",
                 min_samples_leaf=1, random_state=0):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.max_features = max_features
        self.min_samples_leaf = min_samples_leaf
        self.random_state = random_state

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)
        n, d = X.shape
        rng = np.random.default_rng(self.random_state)

        if self.max_features == "sqrt":
            mf = max(1, int(np.sqrt(d)))
        elif isinstance(self.max_features, int):
            mf = self.max_features
        else:
            mf = d

        self.trees_ = []
        self.feature_subsets_ = []
        for t in range(self.n_trees):
            # 1. Bootstrap sample (with replacement).
            idx = rng.integers(0, n, size=n)
            X_boot, y_boot = X[idx], y[idx]

            # 2. Random feature subset for this tree (simplified — true RF resamples per node).
            feats = rng.choice(d, size=mf, replace=False)
            X_sub = X_boot[:, feats]

            tree = DecisionTreeClassifierScratch(
                max_depth=self.max_depth,
                min_samples_leaf=self.min_samples_leaf,
            ).fit(X_sub, y_boot)
            self.trees_.append(tree)
            self.feature_subsets_.append(feats)
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        # Get predictions from each tree, then majority vote.
        all_preds = np.zeros((self.n_trees, len(X)), dtype=int)
        for t, (tree, feats) in enumerate(zip(self.trees_, self.feature_subsets_)):
            all_preds[t] = tree.predict(X[:, feats])
        # Mode across trees.
        from scipy import stats
        return stats.mode(all_preds, axis=0, keepdims=False).mode


# Validate.
from sklearn.ensemble import RandomForestClassifier

mine = RandomForestClassifierScratch(n_trees=20, max_depth=5).fit(X_tr, y_tr)
theirs = RandomForestClassifier(n_estimators=20, max_depth=5, random_state=0).fit(X_tr, y_tr)

print(f"Single tree accuracy: {DecisionTreeClassifier(max_depth=5).fit(X_tr, y_tr).score(X_te, y_te):.3f}")
print(f"My RF accuracy:        {(mine.predict(X_te) == y_te).mean():.3f}")
print(f"sklearn RF accuracy:   {theirs.score(X_te, y_te):.3f}")


## 7.4 Hyperparameters

| Hyperparameter | What it does | Typical |
|---|---|---|
| `n_estimators` | Number of trees | 100-500. More = better, diminishing returns. |
| `max_depth` | Per-tree depth limit | `None` (full trees) or 10-20 |
| `max_features` | Features per split | `sqrt(d)` classification, `d/3` regression |
| `min_samples_leaf` | Smoothing | 1-5 |
| `bootstrap` | Use bootstrap sampling? | `True` (always) |
| `oob_score` | Estimate generalization via out-of-bag samples | `True` for "free" CV |

The big knob is `n_estimators`. **More is always better** for variance reduction (asymptotically), at the cost of compute. Stop adding trees when validation accuracy plateaus.

## 7.5 Out-of-bag (OOB) score

Because each tree sees only ~63% of training samples (bootstrap), the other ~37% can be used as a "free" validation set for that tree. Average the predictions across all trees that didn't see each sample → an unbiased estimate of generalization error without a separate validation set.

In [ ]:
# OOB score example.
rf_oob = RandomForestClassifier(n_estimators=200, oob_score=True, random_state=0).fit(X_dt, y_dt)
print(f"OOB score:        {rf_oob.oob_score_:.4f}")
# This is approximately equal to what you'd get from 5-fold CV.
print(f"Train accuracy:   {rf_oob.score(X_dt, y_dt):.4f}")
# CV the same model for comparison.
from sklearn.model_selection import cross_val_score
cv_score = cross_val_score(
    RandomForestClassifier(n_estimators=200, random_state=0), X_dt, y_dt, cv=5
).mean()
print(f"5-fold CV score:  {cv_score:.4f}")
# OOB and CV should be very close.


## 7.6 Feature importance — gini vs permutation

In [ ]:
from sklearn.inspection import permutation_importance

rf = RandomForestClassifier(n_estimators=200, random_state=0).fit(X_tr, y_tr)

# Method 1: built-in (mean impurity decrease)
imp_builtin = pd.Series(rf.feature_importances_,
                         index=[f"x{i}" for i in range(X_tr.shape[1])])

# Method 2: permutation importance (more robust, model-agnostic)
perm = permutation_importance(rf, X_te, y_te, n_repeats=10, random_state=0)
imp_perm = pd.Series(perm.importances_mean,
                      index=[f"x{i}" for i in range(X_tr.shape[1])])

# Plot both.
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
imp_builtin.sort_values().plot.barh(ax=axes[0], color="steelblue")
axes[0].set_title("Built-in importance (mean impurity decrease)")
axes[0].set_xlabel("Importance")

imp_perm.sort_values().plot.barh(ax=axes[1], color="darkgreen")
axes[1].set_title("Permutation importance (test-set)")
axes[1].set_xlabel("Mean decrease in accuracy")

plt.tight_layout()
plt.show()


**Why permutation importance is better:**
- Model-agnostic (works for any fitted model)
- Computed on **held-out** data → reflects generalization, not training fit
- Not biased toward high-cardinality features
- Captures interaction effects through the model

**Caveat for both:** importance scores for **correlated features** are unreliable. If `x_1` and `x_2` are highly correlated, the model might attribute all importance to one and none to the other, even though both are equally predictive. Drop one or use SHAP for a fairer attribution.

## 7.7 Cheat table

In [ ]:
rf_cheat = pd.DataFrame([
    ["Random Forest", "O(T · n · sqrt(d) · log n)", "O(T · log n)",
     "No", "Sklearn: no (use imputation). LightGBM-style: yes", "Medium (feature importance, but trees are black-box ensemble)"],
], columns=["Algorithm", "Train time", "Predict time", "Needs scaling?", "Handles missing?", "Interpretability"])
rf_cheat


## 7.8 Interview Q&A

**Q: What's the difference between bagging and Random Forest?**
> Bagging = bootstrap aggregating; train models on bootstrap samples of data, average predictions. Random Forest = bagging + **random feature subsampling at each split**. The feature subsampling is what decorrelates the trees and reduces ensemble variance more than bagging alone.

**Q: Does Random Forest overfit?**
> Less than a single deep tree, but yes — especially with too few trees, or with noisy features that the trees latch onto. **Adding more trees does NOT cause overfitting** (it only reduces variance), but each individual tree can still overfit, and that overfitting averages out incompletely if T is small. With T=500+ and `max_depth` limited, RF rarely overfits seriously.

**Q (trap): If I add more trees to a Random Forest, does accuracy always go up?**
> No — it plateaus. Beyond ~200-500 trees, additional trees provide negligible variance reduction (because the average correlation $\rho$ is the asymptotic floor). But adding trees never makes test accuracy worse, only training time worse.

**Q: Why is `max_features = sqrt(d)` the recommendation for classification?**
> Empirical default that balances tree-level accuracy (need enough features to find good splits) against ensemble decorrelation (need few enough features that trees see different ones). Originally from Breiman's 2001 paper; still the standard. For regression, `d/3` is the equivalent default.

**Q (trap): I have a feature with cardinality 1000 (city names). Random Forest's `feature_importances_` says it's the most important feature. Should I trust it?**
> Be skeptical. The built-in (Gini-based) feature importance is biased toward high-cardinality features because they offer more split candidates. Use **permutation importance** on held-out data instead — it's not biased this way. If permutation importance also rates it highly, then trust the result.

**Q: How does Random Forest handle missing values?**
> sklearn's RF does NOT handle missing values — you have to impute first. **LightGBM and XGBoost RF implementations do**: they learn a default direction for each split (left or right) for missing values, optimized during training.

**Q: Compare Random Forest to Gradient Boosting.**
> Random Forest = parallel ensemble of independent trees (averaging reduces variance). Gradient Boosting = sequential ensemble where each tree corrects previous trees' errors (reduces bias). RF is more robust to defaults and less prone to overfitting; GBM gives higher peak accuracy but requires careful tuning of learning rate and tree depth. In Kaggle/competitions, GBM (specifically XGBoost/LightGBM) usually wins; in production, RF is often preferred for its robustness.


---
# 8. Gradient Boosting

The other canonical interview implementation. Conceptually: build trees **sequentially**, each correcting the **residual errors** of the previous trees. Mathematically: it's gradient descent in **function space** — each tree is a step in the direction that minimizes the loss.

Modern variants (XGBoost, LightGBM, CatBoost) are the most-used models in industrial ML. They win on tabular data more often than any other algorithm. Understanding vanilla GBM is the prerequisite for explaining what those variants improve.

## 8.1 The intuition

Start with a baseline prediction $F_0(\mathbf{x}) = \text{mean}(y)$ for regression (log-odds for classification).

For $m = 1, 2, \dots, M$:
1. Compute **residuals**: $r_i = y_i - F_{m-1}(\mathbf{x}_i)$ (or, more generally, the negative gradient of the loss).
2. Fit a small regression tree $h_m$ to predict the residuals from $\mathbf{x}$.
3. Update: $F_m(\mathbf{x}) = F_{m-1}(\mathbf{x}) + \eta \cdot h_m(\mathbf{x})$

$\eta$ is the **learning rate** (typically 0.01-0.1). Small $\eta$ + many trees = better generalization than large $\eta$ + few trees.

Final prediction: $F_M(\mathbf{x}) = F_0 + \eta \sum_{m=1}^M h_m(\mathbf{x})$.

## 8.2 The math — gradient descent in function space

For a general loss $L(y, F(\mathbf{x}))$:
$$F_m(\mathbf{x}) = F_{m-1}(\mathbf{x}) - \eta \cdot \frac{\partial L}{\partial F}\Big|_{F = F_{m-1}}$$

For **squared loss** ($L = \frac{1}{2}(y - F)^2$), the gradient is just the negative residual: $-\partial L / \partial F = y - F$. So fitting a tree to residuals **IS** taking a step in the negative gradient direction. Hence "gradient" boosting.

For **log loss** (classification), the gradient is $y - p$ (where $p$ is the predicted probability). Same flavor, different functional form.

## 8.3 From-scratch implementation (for regression)

In [ ]:
class GradientBoostingRegressorScratch:
    '''
    Gradient Boosting for regression with squared loss.

    Each tree is a small "weak learner" that fits the residuals of the current ensemble.

    Time complexity:
      fit:     O(M · n · d · log n) where M = n_estimators
      predict: O(M · log n)
    '''
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)

        # Initial prediction: mean of y.
        self.F0_ = y.mean()
        F = np.full_like(y, self.F0_)

        self.trees_ = []
        for m in range(self.n_estimators):
            # 1. Compute residuals = negative gradient of squared loss.
            residuals = y - F
            # 2. Fit a regression tree to predict the residuals.
            from sklearn.tree import DecisionTreeRegressor
            tree = DecisionTreeRegressor(max_depth=self.max_depth).fit(X, residuals)
            # 3. Update predictions.
            F = F + self.learning_rate * tree.predict(X)
            self.trees_.append(tree)
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        F = np.full(X.shape[0], self.F0_)
        for tree in self.trees_:
            F = F + self.learning_rate * tree.predict(X)
        return F


# Validate against sklearn.
from sklearn.ensemble import GradientBoostingRegressor

X_gb, y_gb = make_regression(n_samples=500, n_features=8, n_informative=5,
                              noise=10.0, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_gb, y_gb, test_size=0.3, random_state=0)

mine = GradientBoostingRegressorScratch(n_estimators=100, learning_rate=0.1, max_depth=3).fit(X_tr, y_tr)
theirs = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=0).fit(X_tr, y_tr)

from sklearn.metrics import mean_squared_error
print(f"My test MSE:      {mean_squared_error(y_te, mine.predict(X_te)):.2f}")
print(f"sklearn test MSE: {mean_squared_error(y_te, theirs.predict(X_te)):.2f}")


## 8.4 Visualizing the learning curve

In [ ]:
# Track test MSE as a function of n_estimators to see the trajectory.
# Refit with many estimators and probe each intermediate stage.
mine = GradientBoostingRegressorScratch(n_estimators=200, learning_rate=0.1, max_depth=3).fit(X_tr, y_tr)

# Compute predictions after each tree.
train_mse, test_mse = [], []
F_train = np.full(X_tr.shape[0], mine.F0_)
F_test  = np.full(X_te.shape[0], mine.F0_)
for tree in mine.trees_:
    F_train = F_train + mine.learning_rate * tree.predict(X_tr)
    F_test  = F_test  + mine.learning_rate * tree.predict(X_te)
    train_mse.append(mean_squared_error(y_tr, F_train))
    test_mse.append(mean_squared_error(y_te, F_test))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(train_mse, label="Train MSE", color="steelblue")
ax.plot(test_mse, label="Test MSE", color="tomato")
ax.set_xlabel("Number of trees")
ax.set_ylabel("MSE")
ax.set_title("Gradient boosting learning curve — overfitting risk grows with M")
ax.legend()
plt.show()

# Look for the U-shape on test set — that's the overfitting starting.


**Read it:** training MSE keeps dropping monotonically. Test MSE drops then plateaus (or rises slightly = overfitting). The "best" $M$ is where test MSE bottoms out. In practice, use **early stopping** — stop adding trees when validation MSE doesn't improve for N iterations.

## 8.5 Key hyperparameters

| Hyperparameter | Effect |
|---|---|
| `n_estimators` (M) | Number of trees |
| `learning_rate` (η) | Shrinkage per step — smaller = need more trees but better generalization |
| `max_depth` | Per-tree depth — 3-8 typical; deeper = more interactions captured per tree |
| `subsample` | Fraction of rows per tree (stochastic gradient boosting) |
| `min_samples_split`, `min_samples_leaf` | Tree-level regularization |

**The classic GBM tuning recipe:**
1. Fix `learning_rate=0.1`, find roughly best `n_estimators` via CV.
2. Tune `max_depth` and `min_samples_leaf`.
3. Tune `subsample` and `colsample_bytree` (in XGB-style libs).
4. Lower `learning_rate` (to ~0.01), proportionally increase `n_estimators`. Final accuracy boost.

## 8.6 Cheat table

In [ ]:
gb_cheat = pd.DataFrame([
    ["Gradient Boosting (sklearn)", "O(M · n · d · log n)", "O(M · log n)",
     "No", "Imputation needed", "Medium (feature importance, partial deps)"],
], columns=["Algorithm", "Train time", "Predict time", "Needs scaling?", "Handles missing?", "Interpretability"])
gb_cheat


## 8.7 Interview Q&A

**Q: What's the difference between bagging (Random Forest) and boosting?**
> RF trains trees **independently in parallel** on bootstrap samples — averaging reduces variance. Boosting trains trees **sequentially**, each correcting the previous ensemble's errors — reduces bias. RF is robust and hard to misconfigure; boosting is more accurate but requires tuning and is prone to overfitting if you're not careful with learning rate + early stopping.

**Q: Why "gradient" boosting? What is the gradient?**
> Each tree fits the **negative gradient of the loss with respect to the predictions** of the current ensemble. For squared loss, that gradient is the residual $y - F(x)$. For log loss in classification, it's $y - p$. So fitting trees to residuals is gradient descent — but in function space (the space of all possible models) instead of parameter space.

**Q (trap): I increased `learning_rate` from 0.1 to 1.0 and accuracy got worse, even though training is faster. Why?**
> Large learning rates make the ensemble take big steps and **overshoot** the local minima of the loss. The model becomes essentially a single bigger tree, losing the benefit of incremental error correction. The general principle: **smaller learning rate + more trees** outperforms larger learning rate + fewer trees, but takes more compute. Don't trade accuracy for speed without measuring.

**Q: Why are GBM trees shallow (max_depth=3-8) when RF trees are deep?**
> RF wants high-variance, low-bias trees (deep) so averaging can reduce variance. GBM wants low-variance, high-bias **weak learners** (shallow) so the ensemble can iteratively correct them. Each GBM tree is supposed to capture only a small piece of the residual; using deep trees in GBM causes overfitting because each tree fits noise.

**Q (trap): I'm using sklearn's GradientBoostingClassifier on a 10M-row dataset and it's taking forever. What should I do?**
> Stop using sklearn's GBM for large data. Use **LightGBM** (histogram-based binning, leaf-wise growth) or **XGBoost** — they're 10-100x faster on large datasets via algorithmic and implementation optimizations. Sklearn's GBM is fine for prototyping and small data; modern variants are mandatory beyond ~100K rows.

**Q: How does GBM do classification (not just regression)?**
> Same algorithm, different loss. For binary classification, use log loss. The "residual" becomes $y - p$ (probability gradient). The ensemble's output $F(x)$ is the log-odds; final prediction is $\sigma(F(x))$. For multi-class, train K models (one per class) on the softmax cross-entropy gradient.

**Q: Why does GBM overfit eventually but RF doesn't?**
> Each new GBM tree explicitly fits the previous ensemble's errors — on training data. With enough trees, you'll fit the noise in training labels. RF averages independent trees, so noise in each cancels out — adding trees can only reduce variance, never increase it.


---
# 9. XGBoost / LightGBM / CatBoost

The modern boosting libraries. All implement gradient boosting on decision trees, but with very different engineering tradeoffs. Each "won Kaggle" at some point. In 2026 production: **LightGBM is the default**, **XGBoost is the legacy default that still works great**, **CatBoost is the right choice when you have many categorical features**.

This section is about **practical usage + what makes each one distinguishing** — not from-scratch implementation (these are millions of lines of engineering).

## 9.1 What they all share

All three:
- Implement gradient boosting on **regression trees** (CART variants)
- Use the **second-order Taylor expansion** of the loss (XGBoost's original innovation), which uses both gradient AND Hessian information per split. Faster convergence than sklearn's first-order GBM.
- Support classification, regression, ranking, and survival
- Have GPU support, multi-machine training
- Expose `eval_set` and `early_stopping_rounds` for early stopping

## 9.2 XGBoost (eXtreme Gradient Boosting)

**The original modern boosting library.** Introduced:
1. **Regularization in the objective** — both L1 and L2 penalties on leaf weights, plus a penalty on the number of leaves. Combats overfitting compared to vanilla GBM.
2. **Second-order Taylor expansion** of the loss — uses both gradient and Hessian information per split, gives a closed-form for the optimal leaf value.
3. **Sparsity-aware split finding** — efficiently handles missing values and sparse features by learning a "default direction" per split.
4. **Approximate split finding** — instead of exhaustive scan over thresholds, use quantile sketches for speed.
5. **Out-of-the-box parallelization** of split finding across features.

**Tree growth: level-wise (depth-first).** Expands all nodes at the same depth before moving deeper.

## 9.3 LightGBM (Microsoft, 2017)

**Engineering innovations over XGBoost:**
1. **Histogram-based binning** — discretize continuous features into ~256 bins, then iterate over bins instead of individual values. Massive speedup, slight accuracy cost.
2. **Leaf-wise (best-first) tree growth** — at each step, grow the leaf that maximally reduces loss, NOT level-wise. Produces deeper, asymmetric trees with the same number of leaves but better fit. Risk: overfitting on small data; set `max_depth` or `num_leaves` to control.
3. **GOSS (Gradient-based One-Side Sampling)** — downsample training data, keeping all high-gradient (high-error) samples + a random subset of low-gradient (low-error) samples. Faster training with negligible accuracy loss.
4. **EFB (Exclusive Feature Bundling)** — bundle mutually-exclusive sparse features (rarely both non-zero) into one feature, reducing effective dimensionality.

**Result:** typically 2-10x faster than XGBoost on the same hardware, sometimes more accurate.

## 9.4 CatBoost (Yandex, 2017)

**Two original ideas:**
1. **Ordered boosting** — to prevent target leakage from gradients, CatBoost computes residuals using only data BEFORE the current sample in a random permutation. Eliminates the "prediction shift" bias that all other GBM libs have. Slow but theoretically cleaner.
2. **Ordered target encoding** for categoricals — instead of training-set target encoding (which leaks), use only data seen so far in a permutation. Built into the library; you just pass `cat_features=[...]`.

**Best in class when:** dataset has many categorical features (cardinality 5-1000s), or when you don't want to do manual one-hot encoding.

**Slow vs LightGBM**, but often more accurate out-of-the-box (less tuning required).

## 9.5 Practical comparison

In [ ]:
# Compare the three on a realistic classification problem.
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
import time

X_big, y_big = make_classification(
    n_samples=10000, n_features=20, n_informative=10,
    n_redundant=5, random_state=42,
)
X_tr, X_te, y_tr, y_te = train_test_split(X_big, y_big, test_size=0.3, random_state=0)

results = []

# XGBoost
import xgboost as xgb
t0 = time.perf_counter()
xgb_clf = xgb.XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    eval_metric="logloss", random_state=0, verbosity=0,
).fit(X_tr, y_tr)
t_xgb = time.perf_counter() - t0
results.append(("XGBoost", t_xgb, accuracy_score(y_te, xgb_clf.predict(X_te)),
                roc_auc_score(y_te, xgb_clf.predict_proba(X_te)[:, 1])))

# LightGBM
import lightgbm as lgbm
t0 = time.perf_counter()
lgb_clf = lgbm.LGBMClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    random_state=0, verbose=-1,
).fit(X_tr, y_tr)
t_lgb = time.perf_counter() - t0
results.append(("LightGBM", t_lgb, accuracy_score(y_te, lgb_clf.predict(X_te)),
                roc_auc_score(y_te, lgb_clf.predict_proba(X_te)[:, 1])))

# CatBoost
import catboost as cb
t0 = time.perf_counter()
cat_clf = cb.CatBoostClassifier(
    iterations=200, max_depth=6, learning_rate=0.1,
    random_state=0, verbose=False,
).fit(X_tr, y_tr)
t_cat = time.perf_counter() - t0
results.append(("CatBoost", t_cat, accuracy_score(y_te, cat_clf.predict(X_te)),
                roc_auc_score(y_te, cat_clf.predict_proba(X_te)[:, 1])))

comp_df = pd.DataFrame(results, columns=["Library", "Train time (s)", "Test accuracy", "Test ROC-AUC"])
comp_df.round(3)


## 9.6 Early stopping — the production-critical feature

In [ ]:
# Demonstrate early stopping with LightGBM (works the same in XGBoost & CatBoost).
# Split off a validation set from training.
X_tr2, X_va, y_tr2, y_va = train_test_split(X_tr, y_tr, test_size=0.2, random_state=0)

lgb_es = lgbm.LGBMClassifier(
    n_estimators=1000,        # ask for a LOT
    learning_rate=0.05,
    max_depth=6,
    random_state=0,
    verbose=-1,
)
lgb_es.fit(
    X_tr2, y_tr2,
    eval_set=[(X_va, y_va)],
    callbacks=[lgbm.early_stopping(stopping_rounds=20)],
)

print(f"Best iteration: {lgb_es.best_iteration_}")
print(f"Test accuracy:  {accuracy_score(y_te, lgb_es.predict(X_te)):.4f}")


**The early stopping pattern (memorize this):**
1. Split training into train + validation.
2. Fit model with `n_estimators` set high (e.g. 1000).
3. Pass `eval_set` and `early_stopping_rounds`.
4. Model stops adding trees when validation metric hasn't improved for N rounds.
5. Predictions automatically use the best iteration (not the last).

This is how you tune `n_estimators` for free, without grid search. **Always use early stopping in production.**

## 9.7 Categorical features — the CatBoost advantage

In [ ]:
# Build a dataset with a high-cardinality categorical feature.
n = 5000
rng = np.random.default_rng(0)
city = rng.choice([f"city_{i}" for i in range(50)], size=n)
age = rng.normal(35, 10, size=n)
# Target depends on city (hashed) and age.
y_cat = ((age > 35).astype(int) + (np.array([hash(c) % 2 for c in city])) > 1).astype(int)

df_cat = pd.DataFrame({"city": city, "age": age, "y": y_cat})

# Option A: manual one-hot encoding for XGBoost/LightGBM.
df_oh = pd.get_dummies(df_cat, columns=["city"], drop_first=False)
X_oh = df_oh.drop(columns="y").values.astype(float)
print(f"After one-hot: {X_oh.shape[1]} features")

# Option B: CatBoost native categorical support — just pass cat_features.
X_native = df_cat[["city", "age"]].values
print(f"Native (CatBoost): 2 features (one is categorical)")

# Train both — LightGBM with one-hot, CatBoost with native.
X_tr, X_te, y_tr, y_te = train_test_split(X_oh, df_cat["y"].values, test_size=0.3, random_state=0)
lgb_oh = lgbm.LGBMClassifier(n_estimators=100, verbose=-1, random_state=0).fit(X_tr, y_tr)

X_tr_n, X_te_n, y_tr_n, y_te_n = train_test_split(X_native, df_cat["y"].values, test_size=0.3, random_state=0)
cat_native = cb.CatBoostClassifier(
    iterations=100, verbose=False, random_state=0, cat_features=[0]  # index of categorical
).fit(X_tr_n, y_tr_n)

print(f"\nLightGBM + one-hot accuracy: {accuracy_score(y_te, lgb_oh.predict(X_te)):.3f}")
print(f"CatBoost native cat accuracy: {accuracy_score(y_te_n, cat_native.predict(X_te_n)):.3f}")


**When CatBoost's native categoricals shine:**
- High cardinality (>50 unique values per column)
- Multiple categorical columns
- You don't want to engineer encodings manually
- You're prone to leakage with target encoding (CatBoost's ordered version is leakage-safe)

**When LightGBM with one-hot works fine:**
- Low cardinality (< 20 unique values)
- Numerical features dominate the predictive power
- You want maximum training speed

LightGBM also has built-in categorical support (`categorical_feature=...`) — better than one-hot for high-cardinality, but doesn't use ordered encoding like CatBoost.

## 9.8 Cheat table — when to use which

In [ ]:
boosting_cheat = pd.DataFrame([
    ["XGBoost",   "Level-wise growth",  "Slowest",    "Yes (default direction)",   "Manual (one-hot)",          "Mature, well-documented; safe default"],
    ["LightGBM",  "Leaf-wise growth",   "Fastest",    "Yes (default direction)",   "Native (categorical_feature)", "Best speed; default for large data"],
    ["CatBoost",  "Symmetric trees",    "Medium",     "Yes (default direction)",   "Native (ordered encoding)", "Many categoricals; less tuning needed"],
], columns=["Library", "Tree growth strategy", "Speed", "Missing values?", "Categorical handling", "When to use"])
boosting_cheat


## 9.9 Interview Q&A

**Q: What's the second-order Taylor expansion in XGBoost?**
> Vanilla GBM uses only the gradient (first derivative) of the loss to compute residuals. XGBoost uses both gradient AND **Hessian** (second derivative). The Hessian gives info about the loss curvature, leading to a closed-form optimal leaf value and faster convergence. For squared loss the Hessian is constant (=1), so the gain is mostly for log loss / Poisson / custom losses.

**Q (trap): If LightGBM uses leaf-wise growth and grows deeper trees, doesn't it overfit?**
> Yes, that's the tradeoff. LightGBM's leaf-wise growth produces deeper, asymmetric trees that fit training data better than level-wise, but overfit more easily — especially on small datasets. The fix: set `num_leaves` (LightGBM's primary control) to a reasonable value (default 31) and use `min_data_in_leaf` and `max_depth`. For very small datasets (<10K rows), LightGBM can actually be **worse** than XGBoost.

**Q: What does CatBoost do that's "ordered" and why?**
> CatBoost computes target encoding and gradients using only the **prefix** of a random permutation of the data. The result: leaves and encodings are computed without seeing a sample's own label, so no leakage of the target into the gradient or the encoding. This is "ordered boosting" / "ordered target encoding." Standard target encoding on training data leaks the target through the encoded value; ordered encoding solves this principally.

**Q (trap): I have 30 categorical columns each with cardinality 1000. Should I use XGBoost with one-hot encoding?**
> No — 30K one-hot columns will be a memory disaster and the model will overfit. Use either: (a) CatBoost with `cat_features=[...]` — designed exactly for this case, or (b) target encoding (with proper out-of-fold) before XGBoost/LightGBM, or (c) frequency / hashing encoding.

**Q: What's `learning_rate`'s relationship to `n_estimators`?**
> Inverse. Halving learning rate doubles the (rough) required number of trees. Best practice: use a small learning rate (0.01-0.05) + early stopping to find the right number of trees automatically. **You should always tune `learning_rate` and `n_estimators` together, not separately.**

**Q (trap): My XGBoost model trains in 5 minutes but the LightGBM equivalent trains in 30 seconds with similar accuracy. Why?**
> LightGBM's **histogram binning** discretizes continuous features into ~256 bins at the start, so split finding iterates over bins instead of values. XGBoost can do this too (set `tree_method="hist"`), but it's not the default. LightGBM also has GOSS sampling that skips low-gradient examples, and leaf-wise growth means fewer total splits computed. Modern XGBoost with `tree_method="hist"` closes most of the gap.

**Q: Which GBM library should I pick by default in 2026?**
> **LightGBM** for tabular data with mostly-numeric features. **CatBoost** if categoricals dominate. **XGBoost** if you need stability, the largest community, or if your platform doesn't have good LightGBM support. Don't agonize — they're all within 1-2% of each other on most problems. Spend time on feature engineering instead.


---
# 10. The end-to-end pipeline

You've now seen 9 algorithms. None of them matter if you can't wrap them in a **leak-free, reproducible pipeline** that goes from raw data → encoded features → cross-validated estimates → final model.

This section covers:
1. Feature encoding — label vs one-hot vs ordinal vs target vs frequency vs hashing
2. Feature scaling — when each scaler is right
3. The `Pipeline` + `ColumnTransformer` pattern
4. K-fold variants — Stratified, Group, TimeSeriesSplit
5. **Leakage in cross-validation** — the #1 production bug
6. Nested CV for hyperparameter selection


## 10.1 Encoding categorical features

The single most-asked feature engineering question in interviews. Here's the decision tree:

In [ ]:
encoding_decision = pd.DataFrame([
    ["Ordinal encoding",          "Categorical with natural order (low/med/high, L1-L5)",
     "Tree models OR linear models if order matters",  "0, 1, 2, ..."],
    ["One-hot encoding",          "Nominal categoricals, low cardinality (< 20)",
     "Linear, NN, SVM",                                "One binary column per category"],
    ["Label encoding (integer)",  "Nominal categoricals + tree-based model ONLY",
     "Trees (rf, xgboost with categorical support)",   "Arbitrary integers"],
    ["Target / mean encoding",    "High-cardinality categoricals (city, zip)",
     "Tree models with OOF safety",                    "Replace with mean target per category"],
    ["Frequency / count encoding","High-cardinality, when rarity is signal",
     "Tree models",                                    "Replace with category's frequency"],
    ["Hashing encoding",          "Very high cardinality, streaming, no inverse needed",
     "Online learning, real-time",                     "Hash into fixed-size buckets"],
    ["Leave-one-out encoding",    "Smoothed target encoding (less variance)",
     "Tree models",                                    "Mean target excluding current row"],
], columns=["Encoding", "Use when", "Compatible models", "What it produces"])
encoding_decision


**The traps:**

1. **Don't label-encode a nominal categorical and feed it to a linear model.** "Hyderabad=0, Bangalore=1, Mumbai=2" implies Mumbai is "twice" Bangalore. Linear models will fit a slope and produce nonsense.

2. **Target encoding leaks if computed on training data and used on the same training data.** Always compute target encoding using **out-of-fold** values (or CatBoost's ordered version), then apply to held-out data.

3. **One-hot encoding explodes for high cardinality** — a column with 10,000 cities becomes 10,000 columns. Memory + overfitting disaster. Use target encoding or hashing instead.

4. **For tree models, label-encoding is often fine** (and faster than one-hot) even for nominal categoricals — trees can split on `feature <= 3.5` to separate the "first 3 cities" from the rest, and find any pattern.

In [ ]:
# Demonstrate target encoding with OOF safety.
from sklearn.model_selection import KFold

def target_encode_oof(df, cat_col, target_col, n_splits=5, smoothing=10):
    '''
    Out-of-fold target encoding with Bayesian smoothing.

    For each fold's validation rows, compute the target mean per category
    using ONLY the training fold rows.
    Smoothing pulls rare-category means toward the global mean.

    Formula: encoded = (n_cat * mean_cat + smoothing * global_mean) / (n_cat + smoothing)
    '''
    encoded = pd.Series(np.nan, index=df.index)
    global_mean = df[target_col].mean()
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=0)

    for tr_idx, va_idx in kf.split(df):
        # Compute per-category statistics on training fold only.
        agg = df.iloc[tr_idx].groupby(cat_col)[target_col].agg(["mean", "count"])
        agg["smoothed"] = (
            agg["count"] * agg["mean"] + smoothing * global_mean
        ) / (agg["count"] + smoothing)
        # Map onto validation fold.
        encoded.iloc[va_idx] = df.iloc[va_idx][cat_col].map(agg["smoothed"]).fillna(global_mean)
    return encoded


# Synthetic test.
np.random.seed(0)
n = 1000
demo = pd.DataFrame({
    "city": np.random.choice(["Hyd", "Blr", "Mum", "Del", "Pune"], size=n,
                              p=[0.4, 0.3, 0.15, 0.1, 0.05]),
    "y":    np.random.binomial(1, 0.3, size=n),
})
# Make Hyd more positive than the rest.
demo.loc[demo["city"] == "Hyd", "y"] = np.random.binomial(1, 0.7, size=(demo["city"] == "Hyd").sum())

demo["city_oof_te"] = target_encode_oof(demo, "city", "y")
print(demo.groupby("city").agg(true_rate=("y", "mean"), oof_te_mean=("city_oof_te", "mean")).round(3))


## 10.2 Feature scaling — when does it matter?

In [ ]:
scaling_table = pd.DataFrame([
    ["No scaling needed",   "Decision trees, Random Forest, all GBMs",  "Trees split on thresholds; scale-invariant"],
    ["StandardScaler",      "Linear models (when regularized), Logistic, SVM, kNN, PCA, NN",
     "Zero mean, unit variance. Default choice."],
    ["MinMaxScaler",        "When features have known bounded ranges and you need [0,1]",
     "Maps to [0, 1]. Sensitive to outliers."],
    ["RobustScaler",        "When outliers are present",
     "Uses median and IQR instead of mean and std. Outlier-resistant."],
    ["QuantileTransformer", "When distribution is heavy-tailed and shape matters",
     "Maps to uniform or normal via empirical quantiles. Strongest transform."],
    ["No scaling, log transform", "Right-skewed positive features (salary, prices)",
     "Reduces skew, often makes linear models work better"],
], columns=["Strategy", "When to use", "Notes"])
scaling_table


**The single most-tested scaling fact:** trees don't need scaling. Period. Linear models, kNN, SVMs, neural networks **do** need scaling. Memorize this dichotomy.

## 10.3 The `Pipeline` + `ColumnTransformer` pattern

The right way to build a feature pipeline. Encapsulates encoding + scaling + model into one object that can be cross-validated and serialized without leakage.

In [ ]:
# Build a realistic dataset with mixed types.
rng = np.random.default_rng(7)
n = 1000

df_mixed = pd.DataFrame({
    "age":         rng.integers(22, 65, size=n),
    "salary":      rng.lognormal(11, 0.5, size=n),
    "tenure":      rng.uniform(0, 20, size=n),
    "department":  rng.choice(["Eng", "Sales", "HR", "Mkt"], size=n, p=[0.4, 0.3, 0.15, 0.15]),
    "level":       pd.Categorical(rng.choice(["L1", "L2", "L3", "L4"], size=n),
                                   categories=["L1", "L2", "L3", "L4"], ordered=True),
    "remote":      rng.choice([True, False], size=n),
})
df_mixed["left"] = ((df_mixed["salary"] < 60000) | (df_mixed["tenure"] < 2)).astype(int)

# Inject some missing values.
miss_idx = rng.choice(n, size=50, replace=False)
df_mixed.loc[miss_idx, "salary"] = np.nan

print(df_mixed.dtypes)
print(f"\nFirst 5 rows:\n{df_mixed.head()}")


In [ ]:
# Define separate transformation logic for each feature type.
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

numeric_features = ["age", "salary", "tenure"]
nominal_features = ["department"]
ordinal_features = ["level"]
binary_features  = ["remote"]

# Numeric pipeline: impute, then scale.
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])

# Nominal pipeline: one-hot encode.
nominal_pipe = Pipeline([
    ("onehot",  OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

# Ordinal pipeline: ordinal-encode with explicit category order.
ordinal_pipe = Pipeline([
    ("ordinal", OrdinalEncoder(categories=[["L1", "L2", "L3", "L4"]])),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", nominal_pipe, nominal_features),
    ("ord", ordinal_pipe, ordinal_features),
    ("bin", "passthrough",  binary_features),
])

# Wrap with model.
pipe = Pipeline([
    ("prep",  preprocessor),
    ("model", LogisticRegression(max_iter=1000)),
])

# Train.
X = df_mixed.drop(columns="left")
y = df_mixed["left"]

pipe.fit(X, y)
print(f"Pipeline training accuracy: {pipe.score(X, y):.3f}")
print(f"\nTransformed feature shape: {pipe.named_steps['prep'].transform(X).shape}")


**Why pipelines matter for correctness, not just convenience:**

The crucial property: when you call `pipe.fit(X_train, y_train)`, the **scaler learns the mean and std from X_train ONLY**. When you then call `pipe.predict(X_test)`, the scaler applies the **training mean and std** to X_test — it does NOT compute new statistics from test data.

Without a pipeline, it's tempting to do:
```python
X = StandardScaler().fit_transform(X)
X_train, X_test = train_test_split(X)
```
This is **leakage**: the scaler saw the test data's mean and std before training. With a pipeline, you split first, then fit the pipeline on train only.

## 10.4 K-fold cross-validation — the variants

| Variant | When to use |
|---|---|
| **KFold** | Default — random splits, regression with no group structure |
| **StratifiedKFold** | Classification — preserves class proportions in each fold |
| **GroupKFold** | Same group (e.g., same patient) must not appear in train+test together |
| **TimeSeriesSplit** | Temporal data — never train on future, test on past |
| **RepeatedKFold** | Small datasets — repeat CV with different seeds for stability |
| **LeaveOneOut** | Tiny datasets (< 50 samples) — fold = single sample (expensive) |

In [ ]:
# Visualize the four most-used variants.
from sklearn.model_selection import KFold, StratifiedKFold, GroupKFold, TimeSeriesSplit

n_samples = 100
np.random.seed(0)
y_demo = np.array([0]*70 + [1]*30)        # 70/30 class imbalance
groups = np.repeat(np.arange(20), 5)      # 20 groups, 5 samples each

fig, axes = plt.subplots(4, 1, figsize=(11, 7), sharex=True)
splitters = [
    ("KFold (n=5)",            KFold(n_splits=5, shuffle=True, random_state=0), None),
    ("StratifiedKFold (n=5)",  StratifiedKFold(n_splits=5, shuffle=True, random_state=0), None),
    ("GroupKFold (n=5)",       GroupKFold(n_splits=5), groups),
    ("TimeSeriesSplit (n=5)",  TimeSeriesSplit(n_splits=5), None),
]

for ax, (name, splitter, grp) in zip(axes, splitters):
    splits = list(splitter.split(np.arange(n_samples), y_demo, grp) if grp is not None
                  else splitter.split(np.arange(n_samples), y_demo))
    for f, (tr, te) in enumerate(splits):
        mask = np.full(n_samples, "train", dtype="<U10")
        mask[te] = "test"
        ax.scatter(range(n_samples), [f]*n_samples,
                    c=["steelblue" if m == "train" else "tomato" for m in mask],
                    marker="_", lw=8)
    ax.set_yticks(range(len(splits)))
    ax.set_yticklabels([f"fold {i}" for i in range(len(splits))])
    ax.set_title(name)
    ax.set_ylim(-0.5, len(splits) - 0.5)
axes[-1].set_xlabel("Sample index")
plt.tight_layout()
plt.show()


**Read the figure:**
- **KFold:** random splits. Each fold is a contiguous block here because we didn't shuffle.
- **StratifiedKFold:** same as KFold but preserves the 70/30 class ratio in each fold.
- **GroupKFold:** entire groups go to either train or test together. Notice fold boundaries don't cut through groups.
- **TimeSeriesSplit:** **expanding window** — fold 0 trains on first chunk, tests on next; fold 4 trains on most data, tests on last chunk. **Never tests on samples earlier than training.**

## 10.5 Leakage in cross-validation — the #1 bug

The most common production bug: **the pipeline computes some statistic on the full data before splitting.** Examples:
1. **Imputation with mean** computed on full data → test sees train's mean.
2. **Scaling** fit on full data → test sees train's mean and std.
3. **Feature selection** (e.g. SelectKBest) on full data → keep features that happen to look good on test.
4. **Target encoding** without OOF → category encoding leaks the target.
5. **Hyperparameter tuning** on full data and reporting test set accuracy → test set is now used in model selection, not just evaluation.

The fix: **wrap everything in a `Pipeline` and pass it to `cross_val_score`**. The pipeline ensures every transformation is fit only on the training fold inside CV.

In [ ]:
# Demonstrate leakage vs no leakage on a small dataset.
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import cross_val_score

# Generate data where most features are pure noise.
X_leak, y_leak = make_classification(
    n_samples=200, n_features=1000, n_informative=10, n_redundant=0,
    random_state=42,
)

# WRONG: select features on the FULL dataset, then CV.
selector = SelectKBest(f_classif, k=20).fit(X_leak, y_leak)
X_selected = selector.transform(X_leak)
leaky_scores = cross_val_score(LogisticRegression(max_iter=1000), X_selected, y_leak, cv=5)
print(f"LEAKY CV score (feature selection on full data): {leaky_scores.mean():.3f}")

# RIGHT: feature selection INSIDE a pipeline, so it happens per fold.
pipe_correct = Pipeline([
    ("select", SelectKBest(f_classif, k=20)),
    ("model",  LogisticRegression(max_iter=1000)),
])
clean_scores = cross_val_score(pipe_correct, X_leak, y_leak, cv=5)
print(f"CLEAN CV score (feature selection per fold):     {clean_scores.mean():.3f}")
print(f"Gap (the size of the leak):                       {leaky_scores.mean() - clean_scores.mean():.3f}")


**The clean CV score is the realistic estimate of test performance.** The "leaky" score is what you'd report in a paper if you weren't careful, and it would later be unreproducible in production. Always check: does my CV procedure include EVERY data-dependent step inside the fold loop?

## 10.6 Nested cross-validation

For honest hyperparameter selection, you need **two** levels of CV:
- **Outer loop:** estimate generalization performance.
- **Inner loop:** select hyperparameters using cross-validation **within** each outer-loop training fold.

Why: if you select hyperparameters using the same CV that you report scores from, you're **optimizing on the test set** in a subtle way (especially with many hyperparameter combinations). Nested CV gives an unbiased estimate.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Nested CV: outer 5-fold, inner 3-fold for hyperparameter search.
param_grid = {"model__C": [0.01, 0.1, 1, 10]}

inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=0)
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  LogisticRegression(max_iter=1000)),
])

# Inner: GridSearchCV chooses best C per outer fold.
gs = GridSearchCV(pipe, param_grid, cv=inner_cv, scoring="accuracy")

# Outer: cross_val_score evaluates the WHOLE GridSearchCV procedure.
outer_scores = cross_val_score(gs, X_big, y_big, cv=outer_cv, scoring="accuracy")
print(f"Nested CV accuracy: {outer_scores.mean():.4f} ± {outer_scores.std():.4f}")
print("(Inner loop tunes C; outer loop estimates real generalization.)")


**When you need nested CV:**
- Small datasets (< 5000 samples) where a single train/val/test split would be too noisy.
- You're trying many hyperparameter combinations (>20) — the optimism bias of single-CV becomes significant.
- Published / benchmark / publication context.

**When you can skip it:**
- Large datasets where you can afford a proper train/val/test split.
- Simple model with few hyperparameters (logistic regression, plain RF).

For production ML at most companies, a **train/val/test split** (often 60/20/20) with hyperparameter tuning on val and final report on test is fine. Nested CV is for when you need formal statistical rigor.

## 10.7 The full picture — one canonical pipeline

In [ ]:
# Putting it all together: a complete classification pipeline you'd use in an interview.
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
import lightgbm as lgbm

# 1. Identify column types.
numeric_features = ["age", "salary", "tenure"]
categorical_features = ["department"]

# 2. Build the preprocessing block.
preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        # No scaling needed — LightGBM is tree-based.
    ]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot",  OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), categorical_features),
])

# 3. Wrap with model.
clf = Pipeline([
    ("prep",  preprocessor),
    ("model", lgbm.LGBMClassifier(random_state=0, verbose=-1)),
])

# 4. Hyperparameter grid.
param_grid = {
    "model__n_estimators":   [100, 300],
    "model__learning_rate":  [0.05, 0.1],
    "model__max_depth":      [4, 8],
}

# 5. Outer split — hold out test set.
X = df_mixed[numeric_features + categorical_features]
y = df_mixed["left"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)

# 6. Inner CV for hyperparameter tuning.
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=0)
gs = GridSearchCV(clf, param_grid, cv=inner_cv, scoring="roc_auc", n_jobs=-1)
gs.fit(X_train, y_train)

# 7. Report.
print(f"Best CV ROC-AUC:    {gs.best_score_:.4f}")
print(f"Best params:        {gs.best_params_}")
print(f"Held-out test AUC:  {gs.score(X_test, y_test):.4f}")


**This is the template to internalize.** Steps 1-7 are the answer to "walk me through how you'd build a model for this problem" in 95% of interviews:
1. Identify column types.
2. Build a per-column-type preprocessing block via `ColumnTransformer`.
3. Wrap preprocessor + model in a `Pipeline`.
4. Define hyperparameter grid (or random search range).
5. Held-out test set via `train_test_split`.
6. Inner CV for hyperparameter selection via `GridSearchCV`.
7. Report CV-best score AND held-out test score.

If you can write this from memory in any interview, you've answered the systems question.

## 10.8 Interview Q&A on the pipeline

**Q: Why use a Pipeline instead of separately fitting transformers and the model?**
> Two reasons: (1) **correctness** — Pipeline ensures transformations are fit only on the training fold during cross-validation, preventing leakage. (2) **deployment** — you serialize one object instead of multiple; production code calls `pipe.predict(new_data)` and everything happens automatically.

**Q: What's the difference between `StandardScaler` and `MinMaxScaler`? When would you choose each?**
> StandardScaler: subtracts mean, divides by std. Resulting features have mean 0, std 1. **Default choice** for linear models and SVMs. MinMaxScaler: rescales to [0, 1]. Use when you need features in a bounded range (e.g., for some NN layers) or when the data has known min/max. Both are sensitive to outliers; RobustScaler uses median/IQR for outlier resistance.

**Q (trap): I scaled my features before train/test split. Is this a problem?**
> **Yes** — the scaler computed mean/std using test data, so the test set isn't really held out. The scaling is "tainted." For small effects, the bias is small. For aggressive transforms (QuantileTransformer, polynomial features), the bias can be substantial. Always fit transforms inside the pipeline, after splitting.

**Q: When would you use StratifiedKFold vs GroupKFold?**
> StratifiedKFold preserves the **class proportion** in each fold — use for classification with imbalanced classes. GroupKFold ensures samples from the same **group** (patient, user, session) don't appear in both train and test — use when the rows are not independent. They serve different purposes and can be combined (StratifiedGroupKFold).

**Q (trap): My model gets 92% accuracy with 5-fold CV but only 78% on the production test set. What went wrong?**
> Several candidates: (1) **leakage** — train and CV had access to test-derived info. (2) **non-stationarity** — production data has shifted from training. (3) **selection bias** — CV used data sampled differently from production. (4) **target leakage** in features — features used during CV won't be available at prediction time. Run a quick diagnostic: compute KS distance between train features and production features (data drift), and check for any timestamp or post-event features in the model.


---
# Closeout

## What you covered
9 classical algorithms, each with a from-scratch implementation (where reasonable) and library-grade implementation, plus a complete pipeline + CV workflow.

## The interview cheat-sheet for "when do I use what?"

| Problem signature | First reach for |
|---|---|
| Quick baseline, tabular data, want interpretability | Logistic regression / linear regression with regularization |
| Tabular data, no time to tune, robust default | **Random Forest** |
| Tabular data, want maximum accuracy | **LightGBM** with early stopping |
| Tabular data, many high-cardinality categoricals | **CatBoost** |
| Small dataset (<10K rows), clean, want max accuracy | XGBoost or SVM with RBF kernel |
| Text classification, very small training set | Multinomial Naive Bayes |
| Text classification, larger training set | Linear SVM (LinearSVC) on TF-IDF, or logistic regression |
| Need probability calibration | Logistic regression > GBM (use `CalibratedClassifierCV` for SVM) |
| Very simple, near-zero-effort baseline | kNN |
| Streaming / online learning | SGD-based logistic regression, hashing encoding |

## The interview cheat-sheet for "what assumptions matter?"

| Algorithm | Needs scaling? | Sensitive to outliers? | Handles NaN? | Picks features? |
|---|---|---|---|---|
| Linear / Logistic (OLS) | No (yes if regularized) | Yes | No | No (yes with L1) |
| kNN | **Yes** | Yes | No | No |
| Naive Bayes | No | Less (depending on variant) | No | No |
| SVM | **Yes** | Some (less than kNN) | No | No |
| Decision Tree | No | No | Some libraries | Yes (implicit) |
| Random Forest | No | No | Some libraries | Yes (implicit) |
| GBM | No | No | XGB/LGBM/Cat yes | Yes (implicit) |

## What's next
**Notebook 2** in this series covers **unsupervised learning and recommenders** — K-Means, DBSCAN, GMM, hierarchical clustering, PCA, t-SNE/UMAP, TF-IDF, collaborative filtering, matrix factorization.

**Notebook 3** covers **evaluation metrics** comprehensively — classification (binary, multi-class, multi-label), regression, ranking & recommendation, clustering — with the visualizations and the multi-label depth you specifically asked for.

Tell me when you want them.
